In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data"

dbox   = pd.read_csv(os.path.join(DATA_DIR, "fractal_dimensions.csv"))
report = pd.read_csv(os.path.join(DATA_DIR, "sk_all50_report.csv"))

# All tickers in universe
ALL_TICKERS = [
    "AAPL","MSFT","GOOG","AMZN","NVDA","META","TSLA","NFLX","AMD","INTC",
    "JPM","BAC","GS","MS","WFC","BLK","C","AXP","V","MA",
    "JNJ","PFE","UNH","MRK","ABBV","PG","KO","PEP","WMT","COST",
    "XOM","CVX","COP","BA","CAT","GE","HON","LMT","RTX","UPS",
    "GC=F","SI=F","CL=F","BTC-USD","ETH-USD",
    "EWJ","EWZ","FXI","EWG","EWU",
]

Q_VALUES = [10, 50, 75, 100, 150, 200]  # skip q=1 (delta~0)

# ── D_k from report: mean_sk_k2 averaged across windows, per (ticker, t)
# D_k = -mean_sk_k2
dk_base = (report
           .groupby(["ticker", "t"])["mean_sk_k2"]
           .mean()
           .reset_index()
           .rename(columns={"ticker": "Ticker", "mean_sk_k2": "sk_k2"}))
dk_base["D_k"] = -dk_base["sk_k2"]

# ── For tickers with Gamma=0, D_k=0 at all horizons
# Build a complete grid: every ticker × every t in report
all_t_values = sorted(dk_base["t"].unique())

full_grid = pd.MultiIndex.from_product(
    [ALL_TICKERS, all_t_values], names=["Ticker", "t"]
).to_frame(index=False)

dk_full = full_grid.merge(dk_base[["Ticker","t","D_k"]],
                          on=["Ticker","t"], how="left")
dk_full["D_k"] = dk_full["D_k"].fillna(0.0)  # unflagged => D_k=0

# ── Check for bootstrap CI columns (added by the block-bootstrap cell) ──
has_ci = {"D_box_boot_mean", "CI_lo", "CI_hi"}.issubset(dbox.columns)
if not has_ci:
    print("[WARN] No CI columns found in fractal_dimensions.csv — "
          "run the bootstrap cell first. Skipping CI propagation.")

# ── Build lambda_implied for each (ticker, q, t), with CI propagation ──
rows = []
for q in Q_VALUES:
    cols = ["Ticker", "D_box"] + (["CI_lo", "CI_hi"] if has_ci else [])
    dbox_q = dbox[dbox["q"] == q][cols].copy()
    dbox_q["delta"] = dbox_q["D_box"] - 1.0
    if has_ci:
        dbox_q["delta_lo"] = dbox_q["CI_lo"] - 1.0
        dbox_q["delta_hi"] = dbox_q["CI_hi"] - 1.0

    merged = dk_full.merge(dbox_q, on="Ticker", how="left")

    for _, r in merged.iterrows():
        delta = r["delta"]
        D_k   = r["D_k"]
        lam   = (D_k * (q**2) / delta
                 if (not np.isnan(delta) and abs(delta) > 1e-6)
                 else np.nan)

        row = {
            "Ticker":         r["Ticker"],
            "q":              q,
            "t":              r["t"],
            "D_k":            D_k,
            "D_box":          r["D_box"],
            "delta":          delta,
            "lambda_implied": lam,
        }

        if has_ci and "delta_lo" in r.index:
            delta_lo, delta_hi = r["delta_lo"], r["delta_hi"]
            sign_flip = (np.isfinite(delta_lo) and np.isfinite(delta_hi)
                         and (delta_lo * delta_hi < 0))

            lam_at_lo = (D_k * (q**2) / delta_lo
                         if not sign_flip and np.isfinite(delta_lo) and abs(delta_lo) > 1e-6
                         else np.nan)
            lam_at_hi = (D_k * (q**2) / delta_hi
                         if not sign_flip and np.isfinite(delta_hi) and abs(delta_hi) > 1e-6
                         else np.nan)

            row["delta_lo"]           = delta_lo
            row["delta_hi"]           = delta_hi
            row["lambda_ci_lo"]       = np.nanmin([lam_at_lo, lam_at_hi]) if not sign_flip else np.nan
            row["lambda_ci_hi"]       = np.nanmax([lam_at_lo, lam_at_hi]) if not sign_flip else np.nan
            row["delta_ci_sign_flip"] = sign_flip

        rows.append(row)

result = pd.DataFrame(rows)

# ── Flag unreliable (ticker, q) pairs where D_box CI crosses 1.0 ──
if has_ci and "delta_ci_sign_flip" in result.columns:
    flagged = result[result["delta_ci_sign_flip"] == True]
    if not flagged.empty:
        print(f"\n[FLAG] {len(flagged)} (ticker, q, t) rows have D_box CI "
              f"crossing 1.0 — lambda_implied unreliable there:")
        print(flagged[["Ticker", "q", "t", "delta", "delta_lo", "delta_hi"]]
              .drop_duplicates(subset=["Ticker", "q"]).to_string(index=False))
    else:
        print("\n[OK] No (ticker, q) pairs have D_box CI crossing 1.0.")
else:
    flagged = pd.DataFrame()

# ── Per-ticker summary: mean/std/median/min/max across all (q,t) ──
summary = (result
           .groupby("Ticker")["lambda_implied"]
           .agg(["mean","std","median","min","max"])
           .reset_index()
           .rename(columns={
               "mean":   "lambda_mean",
               "std":    "lambda_std",
               "median": "lambda_median",
               "min":    "lambda_min",
               "max":    "lambda_max",
           })
           .sort_values("lambda_mean"))

print("\n── Per-ticker lambda_implied summary (q>1, all t) ──")
print(summary.to_string(index=False))

# ── Per (ticker, q): mean across t ──
pivot_q = (result
           .groupby(["Ticker","q"])["lambda_implied"]
           .mean()
           .reset_index()
           .pivot(index="Ticker", columns="q", values="lambda_implied")
           .reset_index())
pivot_q.columns = ["Ticker"] + [f"lam_q{int(c)}" for c in pivot_q.columns[1:]]

print("\n── Per-ticker lambda_implied by q (mean across t) ──")
print(pivot_q.to_string(index=False))

# ── Per (ticker, t): mean across q — no NaN for zero-Gamma tickers ──
pivot_t = (result
           .groupby(["Ticker","t"])["lambda_implied"]
           .mean()
           .reset_index()
           .pivot(index="Ticker", columns="t", values="lambda_implied")
           .reset_index())

print("\n── Per-ticker lambda_implied by t (mean across q) ──")
print(pivot_t.to_string(index=False))

# ── Save outputs to the data folder ──
result_path  = os.path.join(DATA_DIR, "lambda_implied_full.csv")
summary_path = os.path.join(DATA_DIR, "lambda_implied_summary.csv")
pivot_q_path = os.path.join(DATA_DIR, "lambda_implied_by_q.csv")
pivot_t_path = os.path.join(DATA_DIR, "lambda_implied_by_t.csv")

result.to_csv(result_path, index=False)
summary.to_csv(summary_path, index=False)
pivot_q.to_csv(pivot_q_path, index=False)
pivot_t.to_csv(pivot_t_path, index=False)

if has_ci and not flagged.empty:
    flagged_path = os.path.join(DATA_DIR, "lambda_implied_flagged_ci.csv")
    flagged.to_csv(flagged_path, index=False)
    print(f"\nSaved flagged rows → {flagged_path}")

print(f"\nSaved:\n  {result_path}\n  {summary_path}\n  {pivot_q_path}\n  {pivot_t_path}")


[FLAG] 70 (ticker, q, t) rows have D_box CI crossing 1.0 — lambda_implied unreliable there:
 Ticker   q  t     delta  delta_lo  delta_hi
    AMD 150  1 -0.028009 -0.102639  0.109788
   AMZN 200  1  0.115795 -0.000766  0.174733
   NVDA 200  1  0.077063 -0.077596  0.147136
   META 200  1  0.079434 -0.030773  0.178429
   TSLA 200  1  0.018984 -0.074047  0.146839
   NFLX 200  1  0.008945 -0.082878  0.117089
    AMD 200  1 -0.021279 -0.108050  0.107335
   CL=F 200  1  0.067276 -0.072011  0.121963
ETH-USD 200  1  0.057187 -0.113345  0.130875
    EWZ 200  1  0.096582 -0.058455  0.149405

── Per-ticker lambda_implied summary (q>1, all t) ──
 Ticker  lambda_mean  lambda_std  lambda_median   lambda_min  lambda_max
    AMD  -118.001152  432.868131       0.567045 -2572.757320   68.975520
   GC=F     0.000000    0.000000       0.000000     0.000000    0.000000
    JNJ     0.000000    0.000000       0.000000     0.000000    0.000000
   SI=F     0.000000    0.000000       0.000000     0.000000    0.

In [6]:
# ═══════════════════════════════════════════════════════════════
# GAUNTLET STEP 1: Quadratic vs linear fit of D_box(q)-1 vs q
# Full range (all Q_VALUES) AND restricted range (q<=100, the
# region flagged in the paper as R^2>0.97 / not saturation-contaminated)
# ═══════════════════════════════════════════════════════════════
from numpy.polynomial import polynomial as P

def fit_and_compare(q_arr, y_arr):
    """Fit y = a*q^2 + c (quadratic, no linear term) and y = a'*q + c' (linear).
    Returns dict with coefficients and R^2 for both."""
    q_arr = np.asarray(q_arr, dtype=float)
    y_arr = np.asarray(y_arr, dtype=float)
    mask = np.isfinite(q_arr) & np.isfinite(y_arr)
    q_arr, y_arr = q_arr[mask], y_arr[mask]
    if len(q_arr) < 3:
        return None

    def r2(y_true, y_pred):
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # Quadratic: y = a*q^2 + c
    X_quad = np.column_stack([q_arr ** 2, np.ones_like(q_arr)])
    coef_quad, *_ = np.linalg.lstsq(X_quad, y_arr, rcond=None)
    y_pred_quad = X_quad @ coef_quad
    r2_quad = r2(y_arr, y_pred_quad)

    # Linear: y = a'*q + c'
    X_lin = np.column_stack([q_arr, np.ones_like(q_arr)])
    coef_lin, *_ = np.linalg.lstsq(X_lin, y_arr, rcond=None)
    y_pred_lin = X_lin @ coef_lin
    r2_lin = r2(y_arr, y_pred_lin)

    return {
        "a_quad": coef_quad[0], "c_quad": coef_quad[1], "R2_quad": r2_quad,
        "a_lin": coef_lin[0], "c_lin": coef_lin[1], "R2_lin": r2_lin,
        "delta_R2": r2_quad - r2_lin,
        "n_points": len(q_arr),
    }

fit_rows = []
for ticker in ALL_TICKERS:
    sub = dbox[dbox["Ticker"] == ticker].copy() if "Ticker" in dbox.columns else dbox[dbox["ticker"] == ticker].copy()
    if sub.empty:
        continue
    sub["delta"] = sub["D_box"] - 1.0

    # Full range (all available q, typically includes q=200)
    full_fit = fit_and_compare(sub["q"], sub["delta"])

    # Restricted range: q <= 100, the paper's "clean" region (R^2 > 0.97 for MF fits)
    sub_clean = sub[sub["q"] <= 100]
    clean_fit = fit_and_compare(sub_clean["q"], sub_clean["delta"])

    row = {"Ticker": ticker}
    if full_fit:
        row.update({f"full_{k}": v for k, v in full_fit.items()})
    if clean_fit:
        row.update({f"clean_{k}": v for k, v in clean_fit.items()})
    fit_rows.append(row)

fit_comparison = pd.DataFrame(fit_rows)

print("═" * 70)
print("QUADRATIC vs LINEAR FIT — restricted to q<=100 (clean region)")
print("═" * 70)
cols_show = ["Ticker", "clean_a_quad", "clean_R2_quad", "clean_R2_lin", "clean_delta_R2"]
print(fit_comparison[cols_show].sort_values("clean_delta_R2", ascending=False).to_string(index=False))

n_quad_wins = (fit_comparison["clean_delta_R2"] > 0.01).sum()
n_total = fit_comparison["clean_delta_R2"].notna().sum()
print(f"\nQuadratic beats linear by >0.01 R^2 in {n_quad_wins}/{n_total} assets (clean q<=100 region)")
print(f"Mean R2_quad = {fit_comparison['clean_R2_quad'].mean():.4f}, "
      f"Mean R2_lin = {fit_comparison['clean_R2_lin'].mean():.4f}")

fit_comparison.to_csv(os.path.join(DATA_DIR, "gaspard_quad_vs_linear_fit.csv"), index=False)
print(f"\nSaved → gaspard_quad_vs_linear_fit.csv")

══════════════════════════════════════════════════════════════════════
QUADRATIC vs LINEAR FIT — restricted to q<=100 (clean region)
══════════════════════════════════════════════════════════════════════
 Ticker  clean_a_quad  clean_R2_quad  clean_R2_lin  clean_delta_R2
     GS      0.000012       0.921317      0.999066       -0.077749
    XOM      0.000010       0.920741      0.998956       -0.078215
    FXI      0.000012       0.920618      0.999074       -0.078455
    COP      0.000013       0.919927      0.999034       -0.079107
   META      0.000013       0.917912      0.997076       -0.079165
    AXP      0.000010       0.917603      0.998604       -0.081001
    EWZ      0.000014       0.917791      0.999001       -0.081211
    PFE      0.000009       0.915640      0.998025       -0.082385
    RTX      0.000009       0.914889      0.997680       -0.082792
    HON      0.000009       0.914421      0.997838       -0.083417
   NVDA      0.000016       0.911516      0.995616       -0

In [7]:
# ═══════════════════════════════════════════════════════════════
# GAUNTLET STEP 2: Synthetic validation
# Replicates the paper's exact Phi(t;q) + box-counting pipeline
# (Eqs. 24-30) on processes with KNOWN ground truth.
# ═══════════════════════════════════════════════════════════════

def box_count_dimension(returns, q, n_scales=20, eps_max=0.20):
    """Exact replica of paper Eqs. 24-30: cumulative characteristic-function
    path Phi(t;q), normalized to unit square, box-counted across log-spaced
    scales, D_box = -slope of log N(eps) vs log eps."""
    N = len(returns)
    phase = q * returns
    re = np.cumsum(np.cos(phase))
    im = np.cumsum(np.sin(phase))

    span_x = re.max() - re.min()
    span_y = im.max() - im.min()
    denom = max(span_x, span_y)
    if denom == 0:
        return np.nan, np.nan
    x_tilde = (re - re.min()) / denom
    y_tilde = (im - im.min()) / denom

    eps_min = 2.0 / N
    epsilons = np.logspace(np.log10(eps_min), np.log10(eps_max), n_scales)

    counts = []
    for eps in epsilons:
        ix = np.floor(x_tilde / eps).astype(int)
        iy = np.floor(y_tilde / eps).astype(int)
        occupied = len(set(zip(ix, iy)))
        counts.append(occupied)
    counts = np.array(counts)

    log_eps = np.log(epsilons)
    log_N = np.log(counts)
    valid = np.isfinite(log_N)
    slope, intercept = np.polyfit(log_eps[valid], log_N[valid], 1)
    ss_res = np.sum((log_N[valid] - (slope * log_eps[valid] + intercept)) ** 2)
    ss_tot = np.sum((log_N[valid] - log_N[valid].mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return -slope, r2  # D_box = -slope


def p_model_cascade(n_iter, m1):
    """Deterministic binomial (p-model) measure, paper Eqs. 254-260.
    m1: multiplier in (0, 0.5), controls intermittency. m1=0.5 => monofractal.
    Returns a length-2^n_iter series of increments (log-return proxy)."""
    z = np.array([1.0])
    for _ in range(n_iter):
        z = np.concatenate([m1 * z, (1 - m1) * z])
    # convert measure to a return-like series: center and scale
    r = z - z.mean()
    r = r / r.std()
    return r


# ── (a) GBM / i.i.d. Gaussian null ──
np.random.seed(42)
N_sim = 4096
gbm_returns = np.random.normal(0, 0.01, N_sim)

Q_grid = np.array([10, 50, 75, 100, 150, 200])
gbm_dbox = []
for q in Q_grid:
    d, r2 = box_count_dimension(gbm_returns, q)
    gbm_dbox.append(d)
gbm_dbox = np.array(gbm_dbox)
gbm_fit = fit_and_compare(Q_grid, gbm_dbox - 1.0)

print("═" * 70)
print("SYNTHETIC VALIDATION (a): GBM / i.i.d. Gaussian null")
print("═" * 70)
print(f"D_box(q)-1 across q: {np.round(gbm_dbox - 1.0, 5)}")
print(f"Fitted quadratic coeff a = {gbm_fit['a_quad']:.6f}  (expect ~0)")
print(f"R2_quad = {gbm_fit['R2_quad']:.4f}\n")

# ── (b) p-model cascade, sweeping known intermittency parameter m1 ──
m1_values = [0.5, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]  # 0.5 = monofractal, further from 0.5 = more intermittent
n_iter = 14  # length 2^14 = 16384, comparable to real sample sizes

cascade_rows = []
for m1 in m1_values:
    r_cascade = p_model_cascade(n_iter, m1)
    dbox_vals = []
    for q in Q_grid:
        d, r2 = box_count_dimension(r_cascade, q)
        dbox_vals.append(d)
    dbox_vals = np.array(dbox_vals)
    fit = fit_and_compare(Q_grid, dbox_vals - 1.0)
    cascade_rows.append({
        "m1": m1,
        "intermittency_|m1-0.5|": abs(m1 - 0.5),
        "a_quad": fit["a_quad"],
        "R2_quad": fit["R2_quad"],
        "R2_lin": fit["R2_lin"],
    })

cascade_df = pd.DataFrame(cascade_rows)

print("═" * 70)
print("SYNTHETIC VALIDATION (b): p-model cascade, known intermittency m1")
print("═" * 70)
print(cascade_df.to_string(index=False))

# Monotonicity check: does fitted 'a' increase as m1 moves away from 0.5?
spearman_rho = cascade_df[["intermittency_|m1-0.5|", "a_quad"]].corr(method="spearman").iloc[0, 1]
print(f"\nSpearman correlation (intermittency vs fitted a): {spearman_rho:.4f}")
print("(Values close to +1 support the Gaspard-style correspondence: "
      "greater cascade intermittency -> larger quadratic growth of D_box(q)-1)")

cascade_df.to_csv(os.path.join(DATA_DIR, "gaspard_synthetic_validation.csv"), index=False)
print(f"\nSaved → gaspard_synthetic_validation.csv")

══════════════════════════════════════════════════════════════════════
SYNTHETIC VALIDATION (a): GBM / i.i.d. Gaussian null
══════════════════════════════════════════════════════════════════════
D_box(q)-1 across q: [0.00436 0.04024 0.06217 0.08352 0.13273 0.18897]
Fitted quadratic coeff a = 0.000004  (expect ~0)
R2_quad = 0.9539

══════════════════════════════════════════════════════════════════════
SYNTHETIC VALIDATION (b): p-model cascade, known intermittency m1
══════════════════════════════════════════════════════════════════════
  m1  intermittency_|m1-0.5|        a_quad  R2_quad   R2_lin
0.50                    0.00 -9.374416e-21      NaN      NaN
0.45                    0.05 -1.356116e-06 0.320543 0.147544
0.40                    0.10  4.433462e-07 0.113587 0.035627
0.35                    0.15 -1.658562e-06 0.457645 0.319601
0.30                    0.20 -2.973615e-07 0.028555 0.006976
0.25                    0.25  1.898840e-06 0.754236 0.650618
0.20                    0.30  1.

In [8]:
# ═══════════════════════════════════════════════════════════════
# GAUNTLET STEP 3: Summary verdict — paste-ready for the paper
# ═══════════════════════════════════════════════════════════════
print("═" * 70)
print("GASPARD ANALOGY — VALIDATION SUMMARY")
print("═" * 70)

print(f"""
1. FUNCTIONAL FORM (real data, N={n_total} assets, q<=100 clean region):
   - Quadratic beats linear (delta R2 > 0.01) in {n_quad_wins}/{n_total} assets
   - Mean R2_quad = {fit_comparison['clean_R2_quad'].mean():.4f}
   - Mean R2_lin  = {fit_comparison['clean_R2_lin'].mean():.4f}

2. NULL CASE (GBM synthetic):
   - Fitted quadratic coefficient a = {gbm_fit['a_quad']:.6f} (should be ~0)
   - {'PASS: near-zero growth, consistent with GOOG/SI=F empirical finding' if abs(gbm_fit['a_quad']) < 0.001 else 'CHECK: nonzero growth under null -- investigate'}

3. KNOWN-INTERMITTENCY CASE (p-model cascade sweep):
   - Spearman rho (intermittency vs fitted a) = {spearman_rho:.4f}
   - {'PASS: monotonic relationship supports the correspondence' if spearman_rho > 0.8 else 'WEAK/FAIL: relationship not clearly monotonic -- analogy needs softer framing'}
""")

══════════════════════════════════════════════════════════════════════
GASPARD ANALOGY — VALIDATION SUMMARY
══════════════════════════════════════════════════════════════════════

1. FUNCTIONAL FORM (real data, N=50 assets, q<=100 clean region):
   - Quadratic beats linear (delta R2 > 0.01) in 0/50 assets
   - Mean R2_quad = 0.8929
   - Mean R2_lin  = 0.9922

2. NULL CASE (GBM synthetic):
   - Fitted quadratic coefficient a = 0.000004 (should be ~0)
   - PASS: near-zero growth, consistent with GOOG/SI=F empirical finding

3. KNOWN-INTERMITTENCY CASE (p-model cascade sweep):
   - Spearman rho (intermittency vs fitted a) = 0.5357
   - WEAK/FAIL: relationship not clearly monotonic -- analogy needs softer framing



In [2]:
def fit_power_law(q_arr, y_arr):
    """Fit log|y| = p*log(q) + log(A), i.e. y ~ A*q^p."""
    q_arr = np.asarray(q_arr, dtype=float)
    y_arr = np.asarray(y_arr, dtype=float)
    mask = np.isfinite(q_arr) & np.isfinite(y_arr) & (y_arr > 1e-6) & (q_arr > 0)
    q_arr, y_arr = q_arr[mask], y_arr[mask]
    if len(q_arr) < 3:
        return np.nan, np.nan
    log_q, log_y = np.log(q_arr), np.log(y_arr)
    p, log_A = np.polyfit(log_q, log_y, 1)
    pred = p * log_q + log_A
    ss_res = np.sum((log_y - pred) ** 2)
    ss_tot = np.sum((log_y - log_y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return p, r2

power_rows = []
for ticker in ALL_TICKERS:
    sub = dbox[dbox["Ticker"] == ticker].copy()
    sub = sub[sub["q"] <= 100]
    sub["delta"] = sub["D_box"] - 1.0
    p, r2 = fit_power_law(sub["q"], sub["delta"])
    power_rows.append({"Ticker": ticker, "exponent_p": p, "R2_power": r2})

power_df = pd.DataFrame(power_rows)
print(power_df[["exponent_p", "R2_power"]].describe())
print("\nMedian exponent p:", power_df["exponent_p"].median())

       exponent_p   R2_power
count   50.000000  50.000000
mean     1.130483   0.994773
std      0.202916   0.006236
min      0.789035   0.966858
25%      1.018979   0.994198
50%      1.094096   0.996683
75%      1.187398   0.998301
max      2.094423   0.999741

Median exponent p: 1.0940963413280906


In [3]:
# Quick check: recompute lambda_implied using the fitted exponent p_hat
# instead of q^2, and compare the resulting q-trend to the original.

def lambda_corrected(D_k, q, D_box_minus_1, p_hat):
    if abs(D_box_minus_1) < 1e-6:
        return np.nan
    return D_k * (q ** p_hat) / D_box_minus_1

# Using power_df from the earlier fit (has exponent_p per ticker)
comparison_rows = []
for ticker in ALL_TICKERS:
    p_hat = power_df.loc[power_df["Ticker"] == ticker, "exponent_p"].values
    if len(p_hat) == 0 or np.isnan(p_hat[0]):
        continue
    p_hat = p_hat[0]

    sub = result[result["Ticker"] == ticker]  # from the original lambda_implied cell
    for q in Q_VALUES:
        sub_q = sub[sub["q"] == q]
        if sub_q.empty:
            continue
        mean_lam_orig = sub_q["lambda_implied"].mean()
        # recompute delta and D_k mean for this (ticker, q)
        dbox_row = dbox[(dbox["Ticker"] == ticker) & (dbox["q"] == q)]
        if dbox_row.empty:
            continue
        delta = dbox_row["D_box"].values[0] - 1.0
        dk_mean = sub_q["D_k"].mean()
        lam_corrected = lambda_corrected(dk_mean, q, delta, p_hat)
        comparison_rows.append({
            "Ticker": ticker, "q": q,
            "lambda_orig_q2": mean_lam_orig,
            "lambda_corrected_qp": lam_corrected,
        })

comp_df = pd.DataFrame(comparison_rows)
# Does lambda_corrected still trend upward with q, or does correcting
# the exponent flatten the trend?
trend_orig = comp_df.groupby("q")["lambda_orig_q2"].mean()
trend_corr = comp_df.groupby("q")["lambda_corrected_qp"].mean()
print("Original (q^2) trend across q:\n", trend_orig)
print("\nCorrected (q^p_hat) trend across q:\n", trend_corr)

Original (q^2) trend across q:
 q
10      0.915815
50      3.653586
75      5.722698
100     8.102273
150    10.126128
200    46.044862
Name: lambda_orig_q2, dtype: float64

Corrected (q^p_hat) trend across q:
 q
10     0.120151
50     0.106059
75     0.115189
100    0.126080
150    0.119774
200    0.237711
Name: lambda_corrected_qp, dtype: float64


In [4]:
# ═══════════════════════════════════════════════════════════════
# Corrected lambda_implied table using fitted exponent p_hat
# (replaces the literal q^2 in Eq. lambda_implied with q^p_hat,
#  per-ticker, using exponent_p from power_df / cell 4)
# ═══════════════════════════════════════════════════════════════

SECTOR_MAP = {
    **{t: "Large Cap Technology" for t in
       ["AAPL","MSFT","GOOG","AMZN","NVDA","META","TSLA","NFLX","AMD","INTC"]},
    **{t: "Financials" for t in
       ["JPM","BAC","GS","MS","WFC","BLK","C","AXP","V","MA"]},
    **{t: "Healthcare & Consumer" for t in
       ["JNJ","PFE","UNH","MRK","ABBV","PG","KO","PEP","WMT","COST"]},
    **{t: "Energy & Industrials" for t in
       ["XOM","CVX","COP","BA","CAT","GE","HON","LMT","RTX","UPS"]},
    **{t: "Commodities & Crypto" for t in
       ["GC=F","SI=F","CL=F","BTC-USD","ETH-USD"]},
    **{t: "International ETFs" for t in
       ["EWJ","EWZ","FXI","EWG","EWU"]},
}

def lambda_corrected(D_k, q, D_box_minus_1, p_hat):
    if D_box_minus_1 is None or abs(D_box_minus_1) < 1e-6 or not np.isfinite(p_hat):
        return np.nan
    return D_k * (q ** p_hat) / D_box_minus_1

# ── Build the full (ticker, q) corrected table, mean across horizons t ──
rows_corrected = []
for ticker in ALL_TICKERS:
    p_hat_arr = power_df.loc[power_df["Ticker"] == ticker, "exponent_p"].values
    p_hat = p_hat_arr[0] if len(p_hat_arr) and np.isfinite(p_hat_arr[0]) else np.nan

    sub_ticker = result[result["Ticker"] == ticker]
    for q in Q_VALUES:
        sub_q = sub_ticker[sub_ticker["q"] == q]
        if sub_q.empty:
            continue

        dk_mean       = sub_q["D_k"].mean()
        delta_mean    = sub_q["delta"].mean()          # D_box(q) - 1, mean across t (constant across t anyway)
        lam_orig      = sub_q["lambda_implied"].mean() # literal q^2 version (as in paper Table 8)
        lam_corr      = lambda_corrected(dk_mean, q, delta_mean, p_hat)

        rows_corrected.append({
            "Sector":            SECTOR_MAP.get(ticker, "Other"),
            "Ticker":            ticker,
            "q":                 q,
            "exponent_p":        p_hat,
            "lambda_orig_q2":    lam_orig,
            "lambda_corr_qp":    lam_corr,
        })

corrected_df = pd.DataFrame(rows_corrected)

# ── Cross-sectional growth comparison: literal q^2 vs corrected q^p_hat ──
fold_rows = []
for ticker, sub in corrected_df.groupby("Ticker"):
    sub = sub.sort_values("q")
    if sub["q"].min() > 10 or sub["q"].max() < 200:
        continue
    lo_orig = sub.loc[sub["q"] == sub["q"].min(), "lambda_orig_q2"].values[0]
    hi_orig = sub.loc[sub["q"] == sub["q"].max(), "lambda_orig_q2"].values[0]
    lo_corr = sub.loc[sub["q"] == sub["q"].min(), "lambda_corr_qp"].values[0]
    hi_corr = sub.loc[sub["q"] == sub["q"].max(), "lambda_corr_qp"].values[0]

    fold_orig = (hi_orig / lo_orig) if (np.isfinite(lo_orig) and lo_orig != 0) else np.nan
    fold_corr = (hi_corr / lo_corr) if (np.isfinite(lo_corr) and lo_corr != 0) else np.nan

    fold_rows.append({"Ticker": ticker, "fold_increase_q2": fold_orig, "fold_increase_qp": fold_corr})

fold_df = pd.DataFrame(fold_rows).dropna()

print("═" * 70)
print("LAMBDA_IMPLIED: LITERAL q^2 vs FITTED q^p_hat — CROSS-SECTIONAL SUMMARY")
print("═" * 70)
print(f"\nMedian fold-increase (q=10 -> q=200), literal q^2 definition:   "
      f"{fold_df['fold_increase_q2'].median():.2f}x")
print(f"Median fold-increase (q=10 -> q=200), corrected q^p_hat def.:   "
      f"{fold_df['fold_increase_qp'].median():.2f}x")
print(f"\n(n={len(fold_df)} tickers with valid q=10 and q=200 estimates)")

# ── Per-ticker pivot table, ready for LaTeX (mean lambda_corr_qp by q) ──
pivot_corrected = (corrected_df
                    .pivot_table(index=["Sector", "Ticker"], columns="q",
                                 values="lambda_corr_qp")
                    .reset_index())
pivot_corrected.columns = ["Sector", "Ticker"] + [f"lam_qp_q{int(c)}"
                                                    for c in pivot_corrected.columns[2:]]

print("\n── Per-ticker lambda_implied (q^p_hat corrected) by q ──")
print(pivot_corrected.to_string(index=False))

# ── Save for LaTeX table generation ──
out_path_full  = os.path.join(DATA_DIR, "lambda_corrected_full.csv")
out_path_pivot = os.path.join(DATA_DIR, "lambda_corrected_by_q.csv")
out_path_fold  = os.path.join(DATA_DIR, "lambda_fold_increase_comparison.csv")

corrected_df.to_csv(out_path_full, index=False)
pivot_corrected.to_csv(out_path_pivot, index=False)
fold_df.to_csv(out_path_fold, index=False)

print(f"\nSaved:\n  {out_path_full}\n  {out_path_pivot}\n  {out_path_fold}")

══════════════════════════════════════════════════════════════════════
LAMBDA_IMPLIED: LITERAL q^2 vs FITTED q^p_hat — CROSS-SECTIONAL SUMMARY
══════════════════════════════════════════════════════════════════════

Median fold-increase (q=10 -> q=200), literal q^2 definition:   22.20x
Median fold-increase (q=10 -> q=200), corrected q^p_hat def.:   1.41x

(n=43 tickers with valid q=10 and q=200 estimates)

── Per-ticker lambda_implied (q^p_hat corrected) by q ──
               Sector  Ticker  lam_qp_q10  lam_qp_q50  lam_qp_q75  lam_qp_q100  lam_qp_q150  lam_qp_q200
 Commodities & Crypto BTC-USD    0.080543    0.083434    0.083411     0.078301     0.101993     0.141922
 Commodities & Crypto    CL=F    0.076338    0.067696    0.073644     0.079615     0.122334     0.298209
 Commodities & Crypto ETH-USD    0.095997    0.077886    0.082340     0.109564     0.211379     0.547110
 Commodities & Crypto    GC=F    0.000000    0.000000    0.000000     0.000000     0.000000     0.000000
 Commodit

In [1]:
"""
s_k / D_k Analysis — All 50 Benchmark Tickers, Multiple Windows
================================================================
Runs the single-ticker multi-window analysis across every ticker in the
benchmark universe. Downloads each ticker once (IPO → END_DATE), then
analyses each window independently.

Outputs
-------
sk_plots_all50/<TICKER>/          — per-ticker 4-panel PNG per window
sk_all50_report.csv               — full flagged-horizon table (all tickers)
sk_all50_summary.csv              — one row per (ticker, window): dk_signal,
                                    n_flagged, flagged horizons
sk_all50_heatmap.png              — heatmap: tickers × windows, dk_signal
"""

import os, warnings, time
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from scipy.stats import spearmanr
import yfinance as yf
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════
END_DATE     = "2026-04-11"
WINDOW_SIZES = [500, 1000, 2000, 3000, 5000, 7500, 10000, 15000]

SECTORS = {
    "Tech":                  ["AAPL","MSFT","GOOG","AMZN","NVDA","META","TSLA","NFLX","AMD","INTC"],
    "Financials":            ["JPM","BAC","GS","MS","WFC","BLK","C","AXP","V","MA"],
    "Healthcare & Consumer": ["JNJ","PFE","UNH","MRK","ABBV","PG","KO","PEP","WMT","COST"],
    "Energy & Industrials":  ["XOM","CVX","COP","BA","CAT","GE","HON","LMT","RTX","UPS"],
    "Commodities & Crypto":  ["GC=F","SI=F","CL=F","BTC-USD","ETH-USD"],
    "Intl ETFs":             ["EWJ","EWZ","FXI","EWG","EWU"],
}
ALL_TICKERS  = [t for ts in SECTORS.values() for t in ts]
SECTOR_OF    = {t: s for s, ts in SECTORS.items() for t in ts}

K_VALUES = [0.5, 1.0, 2.0, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]

CF_MIN_MAG                = 0.30
RELATIVE_CHANGE_THRESHOLD = 0.05
ABSOLUTE_CHANGE_THRESHOLD = 5e-5
MIN_VALID_K               = 4
VALIDITY_RATIO            = 0.60
MIN_ABS_MEAN              = 1e-5
CLIP_IQR                  = 3.0

_ALL_CANDIDATE_HORIZONS = [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500, 2000]

OUT_DIR  = r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data\sk_plots_all50"
os.makedirs(OUT_DIR, exist_ok=True)


# ══════════════════════════════════════════════════════════════════
# DERIVED PARAMS
# ══════════════════════════════════════════════════════════════════
def derive_params(window_days):
    max_horizon = window_days // 2
    horizons    = [t for t in _ALL_CANDIDATE_HORIZONS if t <= max_horizon]
    n_origins   = (window_days - 1) - max(horizons)
    return horizons, n_origins


# ══════════════════════════════════════════════════════════════════
# DATA
# ══════════════════════════════════════════════════════════════════
def fetch_ticker(ticker):
    """Download full history. Returns pd.Series of Close prices or None."""
    try:
        df = yf.download(ticker, start="1900-01-01", end=END_DATE,
                         progress=False, auto_adjust=True)
        if df.empty:
            return None
        prices = df["Close"].squeeze().dropna()
        return prices if len(prices) >= 100 else None
    except Exception:
        return None


def slice_window(prices, window_days):
    if len(prices) < window_days:
        return None, None, None
    sliced = prices.iloc[-window_days:]
    return (np.diff(np.log(sliced.values)),
            sliced.index[0].strftime("%Y-%m-%d"),
            sliced.index[-1].strftime("%Y-%m-%d"))


# ══════════════════════════════════════════════════════════════════
# CORE COMPUTATION  (identical to original)
# ══════════════════════════════════════════════════════════════════
def build_ensemble(log_returns, horizons, n_origins):
    displacements = {}
    for t in horizons:
        valid = [log_returns[i + t] - log_returns[i]
                 for i in range(n_origins) if i + t < len(log_returns)]
        displacements[t] = np.array(valid) if valid else np.array([])
    return displacements


def compute_sk(displacements_t, k, t):
    n = len(displacements_t)
    if n < 5:
        return np.nan, n, np.nan
    char_fn   = np.mean(np.exp(1j * k * displacements_t))
    magnitude = np.abs(char_fn)
    if magnitude < 1e-15:
        return np.nan, n, magnitude
    return (1.0 / t) * np.log(magnitude), n, magnitude


def compute_dk(sk, k):
    if np.isnan(sk) or k == 0:
        return np.nan
    return -sk / (k ** 2)


def run_analysis(displacements, horizons):
    results = {}
    for k in K_VALUES:
        results[k] = {}
        for t in horizons:
            sk, n, mag = compute_sk(displacements[t], k, t)
            dk         = compute_dk(sk, k)
            sk_over_k2 = sk / k**2 if not np.isnan(sk) else np.nan
            results[k][t] = (sk, dk, sk_over_k2, n, mag)
    return results


def detect_non_constancy(results, horizons):
    flagged   = []
    n_k_total = len(K_VALUES)
    for t in horizons:
        k_arr   = np.array(K_VALUES)
        sk2_arr = np.array([results[k][t][2] for k in K_VALUES])
        mag_arr = np.array([results[k][t][4] for k in K_VALUES])

        reliable       = (~np.isnan(sk2_arr)) & (~np.isnan(mag_arr)) & (mag_arr >= CF_MIN_MAG)
        vals           = sk2_arr[reliable]
        k_rel          = k_arr[reliable]
        n_rel          = int(reliable.sum())
        coverage_ratio = n_rel / n_k_total

        if n_rel < MIN_VALID_K or coverage_ratio < VALIDITY_RATIO:
            continue

        min_val    = np.min(vals)
        max_val    = np.max(vals)
        abs_change = max_val - min_val
        mean_abs   = np.mean(np.abs(vals))

        if mean_abs > MIN_ABS_MEAN:
            rel_change  = abs_change / mean_abs
            change_type = "relative"
        else:
            rel_change  = abs_change / ABSOLUTE_CHANGE_THRESHOLD
            change_type = "absolute"

        spearman_r, spearman_p = np.nan, np.nan
        if n_rel >= 4:
            spearman_r, spearman_p = spearmanr(k_rel, vals)

        should_flag  = False
        reason_parts = []
        if change_type == "relative" and rel_change > RELATIVE_CHANGE_THRESHOLD:
            should_flag = True
            reason_parts.append(f"rel Δ={rel_change:.1%} > {RELATIVE_CHANGE_THRESHOLD:.0%}")
        elif change_type == "absolute" and abs_change > ABSOLUTE_CHANGE_THRESHOLD:
            should_flag = True
            reason_parts.append(f"abs Δ={abs_change:.2e} > {ABSOLUTE_CHANGE_THRESHOLD:.0e}")

        if should_flag:
            direction = "increasing" if max_val > min_val else "decreasing"
            reason_parts.append(f"({direction} across k)")
            flagged.append({
                "t":          t,
                "n_reliable": n_rel,
                "coverage":   coverage_ratio,
                "mean_sk_k2": float(np.mean(vals)),
                "std_sk_k2":  float(np.std(vals, ddof=1) if n_rel > 1 else 0.0),
                "min_val":    min_val,
                "max_val":    max_val,
                "abs_change": abs_change,
                "rel_change": rel_change if change_type == "relative" else np.nan,
                "spearman_r": spearman_r,
                "spearman_p": spearman_p,
                "reason":     " & ".join(reason_parts),
            })
    return flagged


def dk_signal_from_flagged(flagged, horizons):
    """
    Aggregate non-constancy score: mean rel_change over short-term horizons
    (t <= 20), capped at 5. Returns 0.0 if nothing flagged at short horizons.
    Mirrors the logic used in the benchmark scripts.
    """
    short_vals = []
    for fd in flagged:
        if fd["t"] <= 20:
            rc = fd["rel_change"] if not np.isnan(fd["rel_change"]) else fd["abs_change"] / ABSOLUTE_CHANGE_THRESHOLD
            short_vals.append(float(np.clip(rc, 0., 5.)))
    return float(np.mean(short_vals)) if short_vals else 0.0


# ══════════════════════════════════════════════════════════════════
# PLOT HELPERS  (unchanged from original)
# ══════════════════════════════════════════════════════════════════
def make_k_colors(k_values, cmap_name="turbo"):
    try:    cmap = plt.colormaps[cmap_name]
    except: cmap = cm.get_cmap(cmap_name, max(len(k_values), 2))
    return [cmap(p) for p in np.linspace(0.05, 0.95, len(k_values))]

def make_h_colors(horizons, cmap_name="plasma"):
    try:    cmap = plt.colormaps[cmap_name]
    except: cmap = cm.get_cmap(cmap_name, max(len(horizons), 2))
    return [cmap(p) for p in np.linspace(0.1, 0.9, len(horizons))]

def robust_ylim(values_list, iqr_mult=CLIP_IQR, pad=0.08):
    arr = np.array([v for v in values_list if not np.isnan(v)])
    if len(arr) == 0: return -1, 1
    q1, q3 = np.percentile(arr, [25, 75])
    iqr = q3 - q1
    if iqr == 0: iqr = max(abs(np.ptp(arr)), 1e-10)
    lo, hi = q1 - iqr_mult * iqr, q3 + iqr_mult * iqr
    span = hi - lo if hi > lo else max(abs(hi), 1e-10)
    return lo - pad * span, hi + pad * span

def apply_robust_ylim(ax, values_list):
    ymin, ymax = robust_ylim(values_list)
    ax.set_ylim(ymin, ymax)
    return ymin, ymax

def annotate_cf(ax, x, y, mag, color):
    if not np.isnan(mag) and not np.isnan(y) and mag < 0.50:
        ax.annotate(f"|CF|={mag:.2f}", xy=(x, y), xytext=(0, 8),
                    textcoords="offset points", ha="center", fontsize=6,
                    color=color, alpha=0.85,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white",
                              ec=color, lw=0.5, alpha=0.7))

def place_legend(ax, n_items, fontsize=25):
    ax.legend(fontsize=max(fontsize - 2, 6), loc="upper left",
              bbox_to_anchor=(1.01, 1), borderaxespad=0,
              ncol=max(1, n_items // 12 + 1), framealpha=0.9)

def _add_clipped_markers(ax, xs, ys, color, ymin, ymax):
    for x, y in zip(xs, ys):
        if np.isnan(y): continue
        if y > ymax: ax.plot(x, ymax * 0.98, marker="^", color=color, ms=6, zorder=5, clip_on=False)
        elif y < ymin: ax.plot(x, ymin * 0.98, marker="v", color=color, ms=6, zorder=5, clip_on=False)


# ══════════════════════════════════════════════════════════════════
# PER-WINDOW PLOT  (identical 4-panel layout as original)
# ══════════════════════════════════════════════════════════════════
def plot_results(results, horizons, ticker, window_days, window_start,
                 window_end, flagged_ts, plot_dir):
    plt.rcParams.update({
        "axes.facecolor": "white", "figure.facecolor": "white",
        "axes.edgecolor": "#cccccc", "axes.labelcolor": "#111111",
        "xtick.color": "#333333", "ytick.color": "#333333",
        "text.color": "#111111", "grid.color": "#e5e5e5",
        "grid.linewidth": 0.6, "legend.framealpha": 0.9,
        "legend.edgecolor": "#dddddd",
    })

    n_k      = len(K_VALUES)
    k_colors = make_k_colors(K_VALUES)
    h_colors = make_h_colors(horizons)
    lw = max(0.8, 2.2 - 0.07 * n_k)
    ms = max(2.5, 5.0 - 0.15 * n_k)

    n_rep    = results[K_VALUES[0]][horizons[0]][3]
    flag_str = (f"⚑ Non-constant at t={[f['t'] for f in flagged_ts]}"
                if flagged_ts else "✓ Constant across all t")
    window_label = f"Window: {window_days}d  ({window_start} → {window_end})"

    fig = plt.figure(figsize=(20, 28), facecolor="white")
    '''
    fig.suptitle(
        f"{ticker}  |  {window_label}  |  N={n_rep} origins\n{flag_str}",
        fontsize=11, color="#111111", y=0.998, fontfamily="serif",
    )'''
    gs  = gridspec.GridSpec(4, 1, figure=fig, hspace=0.55)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])
    ax4 = fig.add_subplot(gs[3])

    for ax in [ax1, ax2, ax3, ax4]:
        ax.set_facecolor("white"); ax.grid(True, lw=0.6)
        ax.axhline(0, color="#aaaaaa", lw=0.9, ls="--")
        for spine in ax.spines.values(): spine.set_edgecolor("#cccccc")
        for fd in flagged_ts:
            ax.axvline(fd["t"], color="#e63946", lw=0.8, ls=":", alpha=0.5)

    # Panel 1: s_k/k² vs t
    all_p1 = [results[k][t][2] for k in K_VALUES for t in horizons if not np.isnan(results[k][t][2])]
    ymin1, ymax1 = apply_robust_ylim(ax1, all_p1)
    for i, k in enumerate(K_VALUES):
        ts   = [t for t in horizons if not np.isnan(results[k][t][2])]
        vals = [results[k][t][2] for t in ts]
        mags = [results[k][t][4] for t in ts]
        ax1.plot(ts, vals, color=k_colors[i], lw=lw, marker="o", ms=ms, label=f"k={k}")
        for t, v, mag in zip(ts, vals, mags):
            annotate_cf(ax1, t, v, mag, k_colors[i])
        _add_clipped_markers(ax1, ts, vals, k_colors[i], ymin1, ymax1)
    ax1.set_xlabel(r"Horizon $t$ (days)", fontsize=20)
    ax1.set_ylabel(r"$s_k(t)/k^2$  (days$^{-1}$)", fontsize=20)
    #ax1.set_title(r"$s_k/k^2$ vs $t$" + f"\n[IQR×{CLIP_IQR:.0f} clip]", fontsize=9, fontweight="bold")
    place_legend(ax1, n_k)

    # Panel 2: s_k/k² vs k
    ax2.set_xlim(K_VALUES[0], K_VALUES[-1])
    for j, t in enumerate(horizons):
        rel_mask = np.array([
            not np.isnan(results[k][t][2]) and
            not np.isnan(results[k][t][4]) and
            results[k][t][4] >= CF_MIN_MAG
            for k in K_VALUES])
        vals = np.array([results[k][t][2] for k in K_VALUES])
        mags = np.array([results[k][t][4] for k in K_VALUES])
        ns   = results[K_VALUES[0]][t][3]
        n_display = f"{ns/1000:.1f}k" if ns >= 1000 else str(ns)
        fd   = next((f for f in flagged_ts if f["t"] == t), None)
        lbl  = f"t={t}d (N={n_display})" + (" ⚑" if fd else "")
        ax2.plot(K_VALUES, vals, color=h_colors[j], lw=2, marker="o", ms=5, label=lbl)
        if rel_mask.sum() >= 2:
            ax2.axhline(np.mean(vals[rel_mask]), color=h_colors[j], lw=0.8, ls="--", alpha=0.4)
        for ki, k in enumerate(K_VALUES):
            if not rel_mask[ki] and not np.isnan(vals[ki]):
                ax2.plot(k, vals[ki], "x", color=h_colors[j], ms=8, markeredgewidth=1.5, alpha=0.6)
            annotate_cf(ax2, k, vals[ki], mags[ki], h_colors[j])
    ax2.set_xlabel(r"Wave number $k$", fontsize=20)
    ax2.set_ylabel(r"$s_k(t)/k^2$  (days$^{-1}$)", fontsize=20)
    place_legend(ax2, len(horizons))

    # Panel 3: raw s_k vs t
    all_p3 = [results[k][t][0] for k in K_VALUES for t in horizons if not np.isnan(results[k][t][0])]
    ymin3, ymax3 = apply_robust_ylim(ax3, all_p3)
    for i, k in enumerate(K_VALUES):
        ts   = [t for t in horizons if not np.isnan(results[k][t][0])]
        vals = [results[k][t][0] for t in ts]
        mags = [results[k][t][4] for t in ts]
        ax3.plot(ts, vals, color=k_colors[i], lw=lw, marker="o", ms=ms, label=f"k={k}")
        for t, v, mag in zip(ts, vals, mags):
            annotate_cf(ax3, t, v, mag, k_colors[i])
        _add_clipped_markers(ax3, ts, vals, k_colors[i], ymin3, ymax3)
    ax3.set_xlabel(r"Horizon $t$ (days)", fontsize=20)
    ax3.set_ylabel(r"$s_k(t)$  (days$^{-1}$)", fontsize=20)
    '''ax3.set_title(r"$s_k$ vs $t$ (raw)" + f"\n[IQR×{CLIP_IQR:.0f} clip]", fontsize=9, fontweight="bold")'''
    place_legend(ax3, n_k)

    # Panel 4: raw s_k vs k
    ax4.set_xlim(K_VALUES[0], K_VALUES[-1])
    for j, t in enumerate(horizons):
        vals = np.array([results[k][t][0] for k in K_VALUES])
        mags = np.array([results[k][t][4] for k in K_VALUES])
        ns   = results[K_VALUES[0]][t][3]
        n_display = f"{ns/1000:.1f}k" if ns >= 1000 else str(ns)
        ax4.plot(K_VALUES, vals, color=h_colors[j], lw=2, marker="o", ms=5, label=f"t={t}d (N={n_display})")
        for k, v, mag in zip(K_VALUES, vals, mags):
            annotate_cf(ax4, k, v, mag, h_colors[j])
    ax4.set_xlabel(r"Wave number $k$", fontsize=20)
    ax4.set_ylabel(r"$s_k(t)$  (days$^{-1}$)", fontsize=20)
    #ax4.set_title(r"$s_k$ vs $k$ (raw)", fontsize=10, fontweight="bold")
    place_legend(ax4, len(horizons))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    safe  = ticker.replace("=", "_").replace("-", "_")
    fname = os.path.join(plot_dir, f"{safe}_window{window_days}d.png")
    plt.savefig(fname, dpi=120, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return fname


# ══════════════════════════════════════════════════════════════════
# AGGREGATE HEATMAP  (new — tickers × windows, dk_signal)
# ══════════════════════════════════════════════════════════════════
def plot_heatmap(summary_rows, out_path):
    """
    Heatmap of dk_signal by (ticker, window).
    Rows = tickers (grouped by sector), columns = window sizes.
    Color = dk_signal magnitude; grey = insufficient data.
    """
    tickers_ordered = []
    sector_boundaries = {}
    y = 0
    for sector, tickers in SECTORS.items():
        sector_boundaries[sector] = (y, y + len(tickers))
        tickers_ordered.extend(tickers)
        y += len(tickers)

    windows = sorted(set(r["window_days"] for r in summary_rows))
    n_t, n_w = len(tickers_ordered), len(windows)

    # Build matrix
    dk_matrix = np.full((n_t, n_w), np.nan)
    idx_t = {t: i for i, t in enumerate(tickers_ordered)}
    idx_w = {w: j for j, w in enumerate(windows)}
    for row in summary_rows:
        ti = idx_t.get(row["ticker"])
        wi = idx_w.get(row["window_days"])
        if ti is not None and wi is not None:
            dk_matrix[ti, wi] = row["dk_signal"]

    fig, ax = plt.subplots(figsize=(max(8, n_w * 1.6), max(10, n_t * 0.35)))
    fig.patch.set_facecolor("white"); ax.set_facecolor("#f5f5f5")

    # Mask NaN as grey
    masked = np.ma.masked_invalid(dk_matrix)
    cmap   = plt.cm.YlOrRd.copy(); cmap.set_bad("#cccccc")
    vmax   = max(np.nanpercentile(dk_matrix[~np.isnan(dk_matrix)], 95), 0.1) if not np.all(np.isnan(dk_matrix)) else 1.0
    im     = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=0, vmax=vmax,
                       interpolation="nearest")

    # Cell annotations
    for i in range(n_t):
        for j in range(n_w):
            v = dk_matrix[i, j]
            if not np.isnan(v):
                txt_color = "white" if v > vmax * 0.6 else "#333333"
                ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                        fontsize=7.5, color=txt_color, fontweight="bold")
            else:
                ax.text(j, i, "n/a", ha="center", va="center",
                        fontsize=7, color="#888888")

    # Axes
    ax.set_xticks(range(n_w))
    ax.set_xticklabels([f"{w}d" for w in windows], fontsize=10)
    ax.set_yticks(range(n_t))
    ax.set_yticklabels(tickers_ordered, fontsize=9)
    ax.set_xlabel("Window size (trading days)", fontsize=11)
    '''ax.set_title(
        f"dk_signal by ticker × window  |  end={END_DATE}\n"
        f"Higher = more scale-dependent diffusion (CF non-constant)  |  grey=insufficient data",
        fontsize=11, fontweight="bold", pad=12
    )'''

    # Sector dividers and labels
    for sector, (y0, y1) in sector_boundaries.items():
        mid = (y0 + y1 - 1) / 2
        ax.axhline(y0 - 0.5, color="white", lw=2.5)
        ax.text(-0.7, mid, sector, ha="right", va="center", fontsize=8.5,
                color="#333333", fontweight="bold")

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("dk_signal", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Heatmap saved → {out_path}")


# ══════════════════════════════════════════════════════════════════
# PER-TICKER ANALYSIS
# ══════════════════════════════════════════════════════════════════
def analyse_ticker(ticker, all_prices):
    prices = all_prices.get(ticker)
    if prices is None:
        return [], [], []          # ← now returns 3 things

    ticker_dir = os.path.join(OUT_DIR, ticker.replace("=", "_").replace("-", "_"))
    os.makedirs(ticker_dir, exist_ok=True)

    all_flag_rows = []
    summary_rows  = []
    all_raw_rows  = []              # ← NEW: unfiltered D_k for every k, t

    for window_days in WINDOW_SIZES:
        horizons, n_origins = derive_params(window_days)
        log_returns, window_start, window_end = slice_window(prices, window_days)

        if log_returns is None:
            continue

        needed = max(horizons) + n_origins
        if len(log_returns) < needed:
            continue

        displacements = build_ensemble(log_returns, horizons, n_origins)
        results       = run_analysis(displacements, horizons)

        # ── NEW: capture raw, unfiltered values before flagging discards anything
        for t in horizons:
            sk2_arr = np.array([results[k][t][2] for k in K_VALUES])   # sk_over_k2
            mag_arr = np.array([results[k][t][4] for k in K_VALUES])
            reliable = (~np.isnan(sk2_arr)) & (~np.isnan(mag_arr)) & (mag_arr >= CF_MIN_MAG)
            vals = sk2_arr[reliable]
            all_raw_rows.append({
                "ticker": ticker,
                "window_days": window_days,
                "window_start": window_start,
                "window_end": window_end,
                "t": t,
                "n_reliable": int(reliable.sum()),
                "mean_sk_k2": float(np.mean(vals)) if len(vals) else np.nan,
                "D_k_mean": float(-np.mean(vals)) if len(vals) else np.nan,  # sign convention per your Eq(19)
                "abs_change": float(np.max(vals) - np.min(vals)) if len(vals) else np.nan,
            })

        flagged = detect_non_constancy(results, horizons)
        dk_sig        = dk_signal_from_flagged(flagged, horizons)

        # Per-window plot
        try:
            plot_results(results, horizons, ticker, window_days,
                         window_start, window_end, flagged, ticker_dir)
        except Exception as e:
            print(f"    [WARN] Plot failed for {ticker} w={window_days}: {e}")

        # Collect full flag rows
        for fd in flagged:
            all_flag_rows.append({
                "ticker":       ticker,
                "sector":       SECTOR_OF.get(ticker, "Unknown"),
                "window_days":  window_days,
                "window_start": window_start,
                "window_end":   window_end,
                **fd,
            })

        # Summary row
        summary_rows.append({
            "ticker":          ticker,
            "sector":          SECTOR_OF.get(ticker, "Unknown"),
            "window_days":     window_days,
            "window_start":    window_start,
            "window_end":      window_end,
            "dk_signal":       dk_sig,
            "n_flagged":       len(flagged),
            "flagged_horizons": str([f["t"] for f in flagged]) if flagged else "[]",
            "n_available_days": len(prices),
        })

    return all_flag_rows, summary_rows, all_raw_rows


# ══════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()

    print("=" * 68)
    print("  s_k / D_k ANALYSIS — ALL 50 BENCHMARK TICKERS")
    print("=" * 68)
    print(f"  End date     : {END_DATE}")
    print(f"  Windows      : {WINDOW_SIZES}")
    print(f"  Tickers      : {len(ALL_TICKERS)}")
    print(f"  K values     : {K_VALUES}")
    print(f"  CF_MIN_MAG   : {CF_MIN_MAG}")
    print(f"  REL threshold: {RELATIVE_CHANGE_THRESHOLD:.0%}")
    print("=" * 68)

    # Print derived params table once
    print(f"\n  {'WINDOW':>8}  {'Log-ret':>8}  {'MaxHorizon':>12}  {'N_ORIGINS':>10}  HORIZONS")
    print(f"  {'─'*72}")
    for w in WINDOW_SIZES:
        h, n = derive_params(w)
        print(f"  {w:>8}  {w-1:>8}  {max(h):>12}  {n:>10}  {h}")
    print(f"  {'─'*72}\n")

    # ── Batch download all 50 tickers ─────────────────────────────
    print("Downloading all 50 tickers (full history)...")
    all_prices = {}
    for sector, tickers in SECTORS.items():
        print(f"\n  [{sector}]")
        for ticker in tickers:
            prices = fetch_ticker(ticker)
            if prices is not None:
                all_prices[ticker] = prices
                print(f"    {ticker:<12}  {len(prices):>5} days  "
                      f"{prices.index[0].strftime('%Y-%m-%d')} → {prices.index[-1].strftime('%Y-%m-%d')}")
            else:
                print(f"    {ticker:<12}  FAILED")

    print(f"\n  Downloaded: {len(all_prices)}/{len(ALL_TICKERS)} tickers\n")

    # ── Analyse all tickers ───────────────────────────────────────
    all_flag_rows_global = []
    all_summary_rows     = []
    all_raw_rows_global  = []

    for sector, tickers in SECTORS.items():
        print(f"\n{'─'*68}")
        print(f"  SECTOR: {sector}")
        print(f"{'─'*68}")

        for ticker in tickers:
            t_start = time.time()
            print(f"  {ticker:<12}", end=" ", flush=True)

            flag_rows, summary_rows, raw_rows = analyse_ticker(ticker, all_prices)
            all_flag_rows_global.extend(flag_rows)
            all_summary_rows.extend(summary_rows)
            all_raw_rows_global.extend(raw_rows)

            # One-line per-ticker summary
            if summary_rows:
                parts = []
                for row in sorted(summary_rows, key=lambda r: r["window_days"]):
                    parts.append(f"{row['window_days']}d:dk={row['dk_signal']:.3f}(n={row['n_flagged']})")
                print("  " + "  ".join(parts) + f"  ({time.time()-t_start:.0f}s)")
            else:
                avail = len(all_prices.get(ticker, []))
                print(f"  SKIP (only {avail} days available)  ({time.time()-t_start:.0f}s)")

    # ── Save CSVs ─────────────────────────────────────────────────
    print(f"\n{'='*68}")
    print("  Saving outputs...")

    if all_flag_rows_global:
        df_full = pd.DataFrame(all_flag_rows_global)
        df_full = df_full.sort_values(["sector","ticker","window_days","t"]).reset_index(drop=True)
        df_full.to_csv("sk_all50_report.csv", index=False, float_format="%.6f")
        print(f"  Full report  → sk_all50_report.csv  ({len(df_full)} flagged rows)")
    else:
        print("  Full report  → (no flagged horizons found)")

    if all_raw_rows_global:
        df_raw = pd.DataFrame(all_raw_rows_global)
        df_raw.to_csv("sk_all50_raw.csv", index=False, float_format="%.8f")
        print(f"  Raw report   → sk_all50_raw.csv  ({len(df_raw)} rows)")
        
    if all_summary_rows:
        df_sum = pd.DataFrame(all_summary_rows)
        df_sum = df_sum.sort_values(["sector","ticker","window_days"]).reset_index(drop=True)
        df_sum.to_csv("sk_all50_summary.csv", index=False, float_format="%.6f")
        print(f"  Summary      → sk_all50_summary.csv  ({len(df_sum)} rows)")

        # Print summary table sorted by mean dk_signal descending
        mean_dk = (df_sum.groupby("ticker")["dk_signal"]
                         .mean()
                         .sort_values(ascending=False)
                         .reset_index())
        mean_dk.columns = ["ticker", "mean_dk_signal"]
        mean_dk["sector"] = mean_dk["ticker"].map(SECTOR_OF)
        print(f"\n  {'Ticker':<12} {'Sector':<26} {'mean dk_signal':>14}")
        print(f"  {'─'*54}")
        for _, row in mean_dk.iterrows():
            bar = "█" * min(int(row["mean_dk_signal"] * 200), 30)
            print(f"  {row['ticker']:<12} {row['sector']:<26} {row['mean_dk_signal']:>14.4f}  {bar}")

        # Heatmap
        plot_heatmap(all_summary_rows, "sk_all50_heatmap.png")
    else:
        print("  Summary      → (no rows)")

    elapsed = time.time() - t0
    print(f"\n  Plots   → {OUT_DIR}/<TICKER>/")
    print(f"  Total wall time: {elapsed:.0f}s  ({elapsed/60:.1f} min)")
    print("=" * 68)
    print("Done.")


if __name__ == "__main__":
    main()

  s_k / D_k ANALYSIS — ALL 50 BENCHMARK TICKERS
  End date     : 2026-04-11
  Windows      : [500, 1000, 2000, 3000, 5000, 7500, 10000, 15000]
  Tickers      : 50
  K values     : [0.5, 1.0, 2.0, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
  CF_MIN_MAG   : 0.3
  REL threshold: 5%

    WINDOW   Log-ret    MaxHorizon   N_ORIGINS  HORIZONS
  ────────────────────────────────────────────────────────────────────────
       500       499           250         249  [1, 5, 10, 20, 40, 60, 120, 250]
      1000       999           500         499  [1, 5, 10, 20, 40, 60, 120, 250, 500]
      2000      1999          1000         999  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000]
      3000      2999          1500        1499  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500]
      5000      4999          2000        2999  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500, 2000]
      7500      7499          2000        5499  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500, 2000]

In [2]:
"""
SCI Threshold Sensitivity Analysis
====================================
Reuses fetch_ticker / build_ensemble / run_analysis from the main SCI cell
(cell 1) — those are threshold-independent, so results are computed ONCE
per (ticker, window) and cached. Only the flagging/aggregation step, which
is cheap, is re-run under each alternative threshold setting.

Produces:
  sci_sensitivity_rhat.csv   — Γ̄ per ticker under baseline + each alt config
  sci_sensitivity_summary.csv — Spearman rho vs baseline for each config
"""

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from itertools import product

# ── Baseline (paper) values ─────────────────────────────────────────
BASE_REL_THRESH   = RELATIVE_CHANGE_THRESHOLD   # 0.05
BASE_CF_MIN_MAG   = CF_MIN_MAG                  # 0.30
BASE_MIN_VALID_K  = MIN_VALID_K                 # 4
BASE_VALID_RATIO  = VALIDITY_RATIO              # 0.60

# ── One-at-a-time sweep grid ────────────────────────────────────────
SWEEP = {
    "rel_thresh":  [0.03, 0.05, 0.10],
    "cf_min_mag":  [0.25, 0.30, 0.40],
    "min_valid_k": [3, 4, 5],
    "validity_ratio": [0.50, 0.60, 0.70],
}

# ── Parametrised versions of the two threshold-dependent functions ──
def detect_non_constancy_param(results, horizons, rel_thresh, cf_min_mag,
                                 min_valid_k, validity_ratio,
                                 abs_thresh=ABSOLUTE_CHANGE_THRESHOLD,
                                 min_abs_mean=MIN_ABS_MEAN):
    flagged = []
    n_k_total = len(K_VALUES)
    for t in horizons:
        k_arr   = np.array(K_VALUES)
        sk2_arr = np.array([results[k][t][2] for k in K_VALUES])
        mag_arr = np.array([results[k][t][4] for k in K_VALUES])

        reliable = (~np.isnan(sk2_arr)) & (~np.isnan(mag_arr)) & (mag_arr >= cf_min_mag)
        vals   = sk2_arr[reliable]
        n_rel  = int(reliable.sum())
        coverage_ratio = n_rel / n_k_total

        if n_rel < min_valid_k or coverage_ratio < validity_ratio:
            continue

        min_val, max_val = np.min(vals), np.max(vals)
        abs_change = max_val - min_val
        mean_abs   = np.mean(np.abs(vals))

        if mean_abs > min_abs_mean:
            rel_change  = abs_change / mean_abs
            change_type = "relative"
        else:
            rel_change  = abs_change / abs_thresh
            change_type = "absolute"

        should_flag = (
            (change_type == "relative" and rel_change > rel_thresh) or
            (change_type == "absolute" and abs_change > abs_thresh)
        )
        if should_flag:
            flagged.append({
                "t": t,
                "rel_change": rel_change if change_type == "relative" else np.nan,
                "abs_change": abs_change,
            })
    return flagged


def dk_signal_from_flagged_param(flagged, abs_thresh=ABSOLUTE_CHANGE_THRESHOLD):
    short_vals = []
    for fd in flagged:
        if fd["t"] <= 20:
            rc = fd["rel_change"] if not np.isnan(fd["rel_change"]) else fd["abs_change"] / abs_thresh
            short_vals.append(float(np.clip(rc, 0., 5.)))
    return float(np.mean(short_vals)) if short_vals else 0.0


# ── Step 1: fetch prices + cache threshold-independent `results` ───
print("Fetching prices and caching results (this is the slow part, runs once)...")
all_prices = {}
results_cache = {}   # (ticker, window_days) -> (results, horizons)

for sector, tickers in SECTORS.items():
    for ticker in tickers:
        prices = fetch_ticker(ticker)
        if prices is None:
            continue
        all_prices[ticker] = prices

        for window_days in WINDOW_SIZES:
            horizons, n_origins = derive_params(window_days)
            log_returns, w_start, w_end = slice_window(prices, window_days)
            if log_returns is None:
                continue
            needed = max(horizons) + n_origins
            if len(log_returns) < needed:
                continue

            displacements = build_ensemble(log_returns, horizons, n_origins)
            results = run_analysis(displacements, horizons)
            results_cache[(ticker, window_days)] = (results, horizons)

print(f"Cached {len(results_cache)} (ticker, window) results.\n")


# ── Step 2: compute ticker-level Γ̄ under a given threshold config ──
def compute_rhat_all_tickers(rel_thresh, cf_min_mag, min_valid_k, validity_ratio):
    rhat = {}
    for ticker in all_prices:
        gammas = []
        for window_days in WINDOW_SIZES:
            key = (ticker, window_days)
            if key not in results_cache:
                continue
            results, horizons = results_cache[key]
            flagged = detect_non_constancy_param(
                results, horizons, rel_thresh, cf_min_mag, min_valid_k, validity_ratio)
            gammas.append(dk_signal_from_flagged_param(flagged))
        rhat[ticker] = float(np.mean(gammas)) if gammas else 0.0
    return rhat


# ── Step 3: baseline + one-at-a-time sweep, Spearman rho vs baseline ─
baseline = compute_rhat_all_tickers(BASE_REL_THRESH, BASE_CF_MIN_MAG,
                                     BASE_MIN_VALID_K, BASE_VALID_RATIO)
tickers_order = list(baseline.keys())
baseline_vec  = [baseline[t] for t in tickers_order]

rows = [{"config": "baseline", "param": "—", "value": "—",
         "spearman_rho_vs_baseline": 1.0}]
rhat_table = {"ticker": tickers_order, "baseline": baseline_vec}

for param_name, values in SWEEP.items():
    for v in values:
        cfg = dict(rel_thresh=BASE_REL_THRESH, cf_min_mag=BASE_CF_MIN_MAG,
                   min_valid_k=BASE_MIN_VALID_K, validity_ratio=BASE_VALID_RATIO)
        cfg[param_name] = v
        if cfg == dict(rel_thresh=BASE_REL_THRESH, cf_min_mag=BASE_CF_MIN_MAG,
                        min_valid_k=BASE_MIN_VALID_K, validity_ratio=BASE_VALID_RATIO):
            continue  # skip the baseline value itself, already have it

        rhat = compute_rhat_all_tickers(**cfg)
        vec  = [rhat[t] for t in tickers_order]
        rho, pval = spearmanr(baseline_vec, vec)

        col_name = f"{param_name}={v}"
        rhat_table[col_name] = vec
        rows.append({
            "config": col_name, "param": param_name, "value": v,
            "spearman_rho_vs_baseline": rho, "p_value": pval,
        })
        print(f"{param_name:>15} = {v:<6}  Spearman rho vs baseline = {rho:.4f}  (p={pval:.2e})")

df_summary = pd.DataFrame(rows)
df_summary.to_csv("sci_sensitivity_summary.csv", index=False)

df_rhat = pd.DataFrame(rhat_table)
df_rhat.to_csv("sci_sensitivity_rhat.csv", index=False)

print("\nSaved: sci_sensitivity_summary.csv, sci_sensitivity_rhat.csv")
print("\nLowest robustness (smallest |rho|):")
print(df_summary.sort_values("spearman_rho_vs_baseline").head(5)
      [["config", "spearman_rho_vs_baseline"]])

Fetching prices and caching results (this is the slow part, runs once)...
Cached 314 (ticker, window) results.

     rel_thresh = 0.03    Spearman rho vs baseline = 0.9461  (p=3.89e-25)
     rel_thresh = 0.1     Spearman rho vs baseline = 0.8918  (p=3.77e-18)
     cf_min_mag = 0.25    Spearman rho vs baseline = 1.0000  (p=0.00e+00)
     cf_min_mag = 0.4     Spearman rho vs baseline = 1.0000  (p=0.00e+00)
    min_valid_k = 3       Spearman rho vs baseline = 1.0000  (p=0.00e+00)
    min_valid_k = 5       Spearman rho vs baseline = 1.0000  (p=0.00e+00)
 validity_ratio = 0.5     Spearman rho vs baseline = 1.0000  (p=0.00e+00)
 validity_ratio = 0.7     Spearman rho vs baseline = 1.0000  (p=0.00e+00)

Saved: sci_sensitivity_summary.csv, sci_sensitivity_rhat.csv

Lowest robustness (smallest |rho|):
            config  spearman_rho_vs_baseline
2   rel_thresh=0.1                  0.891769
1  rel_thresh=0.03                  0.946051
0         baseline                  1.000000
3  cf_min_mag=0.2

In [4]:
"""
SCI Threshold Sensitivity Analysis + Significance Test
========================================================
Extends the original threshold-sweep sensitivity analysis with a proper
Monte Carlo significance test for the Spectral Cumulant Index (SCI, Gamma-bar).

WHY THIS IS NEEDED
------------------
The original sensitivity analysis (Section 5.5.4 / Table 4) shows that the
CROSS-SECTIONAL ORDERING of Gamma-bar is robust to the choice of detection
thresholds (delta*, CF magnitude, min valid k, coverage). That is a
robustness check on specification choice, NOT a significance test. It does
not tell you whether any single ticker's observed Gamma-bar exceeds what
pure finite-sample noise in the empirical characteristic function would
produce if the true return process were Gaussian at every horizon.

This script adds that test, following the same logic already used for
D_box in Table 5 (block-resampling to get a reference distribution rather
than relying on a single point estimate):

  1. NULL MODEL: under H0, log-returns at a given window are iid Gaussian
     with variance matched to the empirical variance over that window.
     Per Eq. (20), this makes D_k(t) = -sigma^2 / 2, exactly constant in
     both k and t. Any apparent non-constancy detected in a simulated
     null series is therefore pure estimation noise from finite N_orig,
     not genuine leptokurtosis.

  2. For each (ticker, window), B synthetic Gaussian series are drawn and
     pushed through the IDENTICAL pipeline used on the real data
     (build_ensemble -> run_analysis -> detect_non_constancy ->
     dk_signal_from_flagged), using the baseline thresholds. This yields
     an empirical null distribution of Gamma for that (ticker, window).

  3. Ticker-level significance: for each replicate b, Gamma is averaged
     across the SAME set of valid windows used to compute the ticker's
     real Gamma-bar (mirroring how Gamma-bar itself is defined), giving a
     null distribution of Gamma-bar. The p-value is the one-sided upper
     tail probability of the null exceeding the observed Gamma-bar.

  4. Benjamini-Hochberg FDR correction is applied across all 50 tickers,
     exactly as done for the Delta-alpha_corr surrogate test in Table 1,
     so the significance claims are held to the same standard.

Outputs
-------
sci_sensitivity_summary.csv     -- unchanged: threshold-sweep robustness (rho vs baseline)
sci_sensitivity_rhat.csv        -- unchanged: per-ticker Gamma-bar under each threshold config
sci_null_by_window.csv          -- NEW: per (ticker, window) observed Gamma, null mean/std, p-value
sci_significance_summary.csv    -- NEW: per-ticker Gamma-bar, p-value, BH q-value, significant flag

NOTE ON COST
------------
This is Monte Carlo, not free. Runtime scales as
    n_tickers * n_windows * B * cost(run_analysis)
Default B=200 (vs B=500 used for D_box in Table 5, which was a single
resample per asset rather than per-window-per-asset). Reduce B for a
quick pass, or parallelize the outer ticker loop with
joblib.Parallel / multiprocessing if you need full B=500 across all 50
tickers x 8 windows in reasonable wall time.
"""

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, false_discovery_control
from itertools import product

# ── Baseline (paper) values ─────────────────────────────────────────
BASE_REL_THRESH   = RELATIVE_CHANGE_THRESHOLD   # 0.05
BASE_CF_MIN_MAG   = CF_MIN_MAG                  # 0.30
BASE_MIN_VALID_K  = MIN_VALID_K                 # 4
BASE_VALID_RATIO  = VALIDITY_RATIO              # 0.60

# ── One-at-a-time sweep grid (unchanged) ────────────────────────────
SWEEP = {
    "rel_thresh":  [0.03, 0.05, 0.10],
    "cf_min_mag":  [0.25, 0.30, 0.40],
    "min_valid_k": [3, 4, 5],
    "validity_ratio": [0.50, 0.60, 0.70],
}

# ── Significance test config ────────────────────────────────────────
N_NULL_REPLICATES = 500      # B; raise to 500 to match Table 5's convention
NULL_SEED         = 42    # for reproducibility
FDR_ALPHA         = 0.05     # BH significance level, matches Table 1


# ── Parametrised threshold-dependent functions (unchanged) ─────────
def detect_non_constancy_param(results, horizons, rel_thresh, cf_min_mag,
                                 min_valid_k, validity_ratio,
                                 abs_thresh=ABSOLUTE_CHANGE_THRESHOLD,
                                 min_abs_mean=MIN_ABS_MEAN):
    flagged = []
    n_k_total = len(K_VALUES)
    for t in horizons:
        k_arr   = np.array(K_VALUES)
        sk2_arr = np.array([results[k][t][2] for k in K_VALUES])
        mag_arr = np.array([results[k][t][4] for k in K_VALUES])

        reliable = (~np.isnan(sk2_arr)) & (~np.isnan(mag_arr)) & (mag_arr >= cf_min_mag)
        vals   = sk2_arr[reliable]
        n_rel  = int(reliable.sum())
        coverage_ratio = n_rel / n_k_total

        if n_rel < min_valid_k or coverage_ratio < validity_ratio:
            continue

        min_val, max_val = np.min(vals), np.max(vals)
        abs_change = max_val - min_val
        mean_abs   = np.mean(np.abs(vals))

        if mean_abs > min_abs_mean:
            rel_change  = abs_change / mean_abs
            change_type = "relative"
        else:
            rel_change  = abs_change / abs_thresh
            change_type = "absolute"

        should_flag = (
            (change_type == "relative" and rel_change > rel_thresh) or
            (change_type == "absolute" and abs_change > abs_thresh)
        )
        if should_flag:
            flagged.append({
                "t": t,
                "rel_change": rel_change if change_type == "relative" else np.nan,
                "abs_change": abs_change,
            })
    return flagged


def dk_signal_from_flagged_param(flagged, abs_thresh=ABSOLUTE_CHANGE_THRESHOLD):
    short_vals = []
    for fd in flagged:
        if fd["t"] <= 20:
            rc = fd["rel_change"] if not np.isnan(fd["rel_change"]) else fd["abs_change"] / abs_thresh
            short_vals.append(float(np.clip(rc, 0., 5.)))
    return float(np.mean(short_vals)) if short_vals else 0.0


# ── Step 1: fetch prices + cache threshold-independent `results` ───
print("Fetching prices and caching results (this is the slow part, runs once)...")
all_prices = {}
results_cache = {}   # (ticker, window_days) -> (results, horizons)
log_returns_cache = {}  # (ticker, window_days) -> log_returns array (needed for sigma matching)

for sector, tickers in SECTORS.items():
    for ticker in tickers:
        prices = fetch_ticker(ticker)
        if prices is None:
            continue
        all_prices[ticker] = prices

        for window_days in WINDOW_SIZES:
            horizons, n_origins = derive_params(window_days)
            log_returns, w_start, w_end = slice_window(prices, window_days)
            if log_returns is None:
                continue
            needed = max(horizons) + n_origins
            if len(log_returns) < needed:
                continue

            displacements = build_ensemble(log_returns, horizons, n_origins)
            results = run_analysis(displacements, horizons)
            results_cache[(ticker, window_days)] = (results, horizons)
            log_returns_cache[(ticker, window_days)] = log_returns

print(f"Cached {len(results_cache)} (ticker, window) results.\n")


# ── Step 2: compute ticker-level Gamma-bar under a given threshold config ──
def compute_rhat_all_tickers(rel_thresh, cf_min_mag, min_valid_k, validity_ratio):
    rhat = {}
    valid_windows_used = {}
    for ticker in all_prices:
        gammas = []
        windows_used = []
        for window_days in WINDOW_SIZES:
            key = (ticker, window_days)
            if key not in results_cache:
                continue
            results, horizons = results_cache[key]
            flagged = detect_non_constancy_param(
                results, horizons, rel_thresh, cf_min_mag, min_valid_k, validity_ratio)
            gammas.append(dk_signal_from_flagged_param(flagged))
            windows_used.append(window_days)
        rhat[ticker] = float(np.mean(gammas)) if gammas else 0.0
        valid_windows_used[ticker] = windows_used
    return rhat, valid_windows_used


# ── Step 3: baseline + one-at-a-time sweep, Spearman rho vs baseline (unchanged) ─
baseline, baseline_windows_used = compute_rhat_all_tickers(
    BASE_REL_THRESH, BASE_CF_MIN_MAG, BASE_MIN_VALID_K, BASE_VALID_RATIO)
tickers_order = list(baseline.keys())
baseline_vec  = [baseline[t] for t in tickers_order]

rows = [{"config": "baseline", "param": "—", "value": "—",
         "spearman_rho_vs_baseline": 1.0}]
rhat_table = {"ticker": tickers_order, "baseline": baseline_vec}

for param_name, values in SWEEP.items():
    for v in values:
        cfg = dict(rel_thresh=BASE_REL_THRESH, cf_min_mag=BASE_CF_MIN_MAG,
                   min_valid_k=BASE_MIN_VALID_K, validity_ratio=BASE_VALID_RATIO)
        cfg[param_name] = v
        if cfg == dict(rel_thresh=BASE_REL_THRESH, cf_min_mag=BASE_CF_MIN_MAG,
                        min_valid_k=BASE_MIN_VALID_K, validity_ratio=BASE_VALID_RATIO):
            continue  # skip the baseline value itself, already have it

        rhat, _ = compute_rhat_all_tickers(**cfg)
        vec  = [rhat[t] for t in tickers_order]
        rho, pval = spearmanr(baseline_vec, vec)

        col_name = f"{param_name}={v}"
        rhat_table[col_name] = vec
        rows.append({
            "config": col_name, "param": param_name, "value": v,
            "spearman_rho_vs_baseline": rho, "p_value": pval,
        })
        print(f"{param_name:>15} = {v:<6}  Spearman rho vs baseline = {rho:.4f}  (p={pval:.2e})")

df_summary = pd.DataFrame(rows)
df_summary.to_csv("sci_sensitivity_summary.csv", index=False)

df_rhat = pd.DataFrame(rhat_table)
df_rhat.to_csv("sci_sensitivity_rhat.csv", index=False)

print("\nSaved: sci_sensitivity_summary.csv, sci_sensitivity_rhat.csv")
print("\nLowest robustness (smallest |rho|):")
print(df_summary.sort_values("spearman_rho_vs_baseline").head(5)
      [["config", "spearman_rho_vs_baseline"]])


# ══════════════════════════════════════════════════════════════════
# STEP 4 (NEW): MONTE CARLO NULL TEST FOR GAMMA-BAR SIGNIFICANCE
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 68)
print("  SIGNIFICANCE TEST: Gamma-bar vs Gaussian-null estimation noise")
print("=" * 68)


def simulate_null_gamma(sigma, n_log_returns, horizons, n_origins, rng):
    """
    Draw one iid Gaussian log-return series matched to the real window's
    sigma and length, push it through the identical detection pipeline
    at baseline thresholds, and return the resulting (null) Gamma.
    Under H0 (Gaussian returns), D_k(t) is exactly constant in k and t
    per Eq. (20); any nonzero Gamma here is pure finite-N noise.
    """
    sim_log_returns = rng.normal(loc=0.0, scale=sigma, size=n_log_returns)
    displacements = build_ensemble(sim_log_returns, horizons, n_origins)
    results = run_analysis(displacements, horizons)
    flagged = detect_non_constancy_param(
        results, horizons,
        BASE_REL_THRESH, BASE_CF_MIN_MAG, BASE_MIN_VALID_K, BASE_VALID_RATIO)
    return dk_signal_from_flagged_param(flagged)


rng = np.random.default_rng(NULL_SEED)

null_by_window_rows = []
# null_gamma_by_ticker_window[ticker][window_days] = np.array of B null Gammas
null_gamma_by_ticker_window = {}

n_keys = len(results_cache)
for i, ((ticker, window_days), (results, horizons)) in enumerate(results_cache.items()):
    log_returns = log_returns_cache[(ticker, window_days)]
    n_origins = (window_days - 1) - max(horizons)
    sigma = float(np.std(log_returns, ddof=1))

    # Observed Gamma at baseline thresholds for this (ticker, window)
    flagged_obs = detect_non_constancy_param(
        results, horizons,
        BASE_REL_THRESH, BASE_CF_MIN_MAG, BASE_MIN_VALID_K, BASE_VALID_RATIO)
    gamma_obs = dk_signal_from_flagged_param(flagged_obs)

    null_gammas = np.array([
        simulate_null_gamma(sigma, len(log_returns), horizons, n_origins, rng)
        for _ in range(N_NULL_REPLICATES)
    ])

    null_gamma_by_ticker_window.setdefault(ticker, {})[window_days] = null_gammas

    p_window = (1 + np.sum(null_gammas >= gamma_obs)) / (N_NULL_REPLICATES + 1)
    null_by_window_rows.append({
        "ticker": ticker,
        "sector": SECTOR_OF.get(ticker, "Unknown"),
        "window_days": window_days,
        "sigma": sigma,
        "gamma_observed": gamma_obs,
        "null_gamma_mean": float(np.mean(null_gammas)),
        "null_gamma_std": float(np.std(null_gammas, ddof=1)),
        "null_gamma_p95": float(np.percentile(null_gammas, 95)),
        "p_value_window": p_window,
    })

    if (i + 1) % 25 == 0 or (i + 1) == n_keys:
        print(f"  null-simulated {i+1}/{n_keys} (ticker, window) pairs...")

df_null_window = pd.DataFrame(null_by_window_rows)
df_null_window = df_null_window.sort_values(["sector", "ticker", "window_days"]).reset_index(drop=True)
df_null_window.to_csv("sci_null_by_window.csv", index=False)
print(f"\n  Per-window null test -> sci_null_by_window.csv  ({len(df_null_window)} rows)")


# ── Ticker-level significance: null distribution of Gamma-bar ──────
# For each replicate b, average the null Gamma across the SAME windows
# used in the real Gamma-bar for that ticker (mirrors the real aggregation).
ticker_sig_rows = []
for ticker in tickers_order:
    windows_used = baseline_windows_used.get(ticker, [])
    if not windows_used:
        continue

    gamma_bar_obs = baseline[ticker]  # already computed with baseline thresholds

    # Stack null Gamma arrays across the used windows -> shape (n_windows, B)
    null_stack = np.array([
        null_gamma_by_ticker_window[ticker][w] for w in windows_used
    ])
    null_gamma_bar_dist = null_stack.mean(axis=0)  # shape (B,)

    p_value = (1 + np.sum(null_gamma_bar_dist >= gamma_bar_obs)) / (N_NULL_REPLICATES + 1)

    ticker_sig_rows.append({
        "ticker": ticker,
        "sector": SECTOR_OF.get(ticker, "Unknown"),
        "gamma_bar_observed": gamma_bar_obs,
        "n_windows_used": len(windows_used),
        "null_gamma_bar_mean": float(np.mean(null_gamma_bar_dist)),
        "null_gamma_bar_std": float(np.std(null_gamma_bar_dist, ddof=1)),
        "p_value": p_value,
    })

df_sig = pd.DataFrame(ticker_sig_rows)

# Benjamini-Hochberg FDR correction across all 50 tickers, same standard as Table 1
df_sig["q_value"] = false_discovery_control(df_sig["p_value"].values, method="bh")
df_sig["significant_q05"] = df_sig["q_value"] < FDR_ALPHA

df_sig = df_sig.sort_values("gamma_bar_observed", ascending=False).reset_index(drop=True)
df_sig.to_csv("sci_significance_summary.csv", index=False)

n_sig = int(df_sig["significant_q05"].sum())
print(f"\n  Ticker-level significance -> sci_significance_summary.csv  ({len(df_sig)} tickers)")
print(f"  {n_sig} of {len(df_sig)} tickers have Gamma-bar significant at FDR q < {FDR_ALPHA}")
print("\n  Top 10 by Gamma-bar, with significance:")
print(df_sig.head(10)[["ticker", "sector", "gamma_bar_observed", "p_value", "q_value", "significant_q05"]]
      .to_string(index=False))

print("\n  Tickers with Gamma-bar > 0 but NOT significant at q < 0.05 (candidates for 'noise, not signal'):")
not_sig_but_nonzero = df_sig[(df_sig["gamma_bar_observed"] > 0) & (~df_sig["significant_q05"])]
if len(not_sig_but_nonzero):
    print(not_sig_but_nonzero[["ticker", "gamma_bar_observed", "p_value", "q_value"]].to_string(index=False))
else:
    print("  (none — every ticker with nonzero Gamma-bar clears the FDR threshold)")

print("\nDone.")

Fetching prices and caching results (this is the slow part, runs once)...
Cached 314 (ticker, window) results.

     rel_thresh = 0.03    Spearman rho vs baseline = 0.9461  (p=3.89e-25)
     rel_thresh = 0.1     Spearman rho vs baseline = 0.8918  (p=3.77e-18)
     cf_min_mag = 0.25    Spearman rho vs baseline = 1.0000  (p=0.00e+00)
     cf_min_mag = 0.4     Spearman rho vs baseline = 1.0000  (p=0.00e+00)
    min_valid_k = 3       Spearman rho vs baseline = 1.0000  (p=0.00e+00)
    min_valid_k = 5       Spearman rho vs baseline = 1.0000  (p=0.00e+00)
 validity_ratio = 0.5     Spearman rho vs baseline = 1.0000  (p=0.00e+00)
 validity_ratio = 0.7     Spearman rho vs baseline = 1.0000  (p=0.00e+00)

Saved: sci_sensitivity_summary.csv, sci_sensitivity_rhat.csv

Lowest robustness (smallest |rho|):
            config  spearman_rho_vs_baseline
2   rel_thresh=0.1                  0.891769
1  rel_thresh=0.03                  0.946051
0         baseline                  1.000000
3  cf_min_mag=0.2

In [4]:
import pandas as pd
raw = pd.read_csv("sk_all50_raw.csv")
seven = ["GOOG","JNJ","PFE","CAT","GC=F","SI=F","EWJ"]
print(raw[raw["ticker"].isin(seven)][["ticker","window_days","t","D_k_mean"]].describe())
print(raw[raw["ticker"].isin(seven)].groupby("ticker")["D_k_mean"].agg(["mean","std","min","max"]))

        window_days            t      D_k_mean
count    488.000000   488.000000  4.880000e+02
mean    4348.360656   327.385246  3.238305e-05
std     3754.197839   502.981190  8.071988e-05
min      500.000000     1.000000  6.000000e-08
25%     2000.000000    10.000000  5.175000e-07
50%     3000.000000    60.000000  3.295000e-06
75%     5000.000000   500.000000  1.887000e-05
max    15000.000000  2000.000000  4.650400e-04
            mean       std           min       max
ticker                                            
CAT     0.000047  0.000106  1.800000e-07  0.000447
EWJ     0.000021  0.000051  8.000000e-08  0.000250
GC=F    0.000013  0.000030  6.000000e-08  0.000142
GOOG    0.000048  0.000109  1.600000e-07  0.000465
JNJ     0.000019  0.000046  6.000000e-08  0.000199
PFE     0.000030  0.000070  1.000000e-07  0.000303
SI=F    0.000050  0.000112  2.200000e-07  0.000464


In [6]:
"""
Φ(t; q) — Full Canvas SVG + Box-Counting Fractal Dimension
=============================================================
- Per-ticker subfolders inside OUT_DIR
- Saves SVG for each ticker × q  (unchanged)
- Saves crisp zoomed PNGs via matplotlib (data-space crop, 300 DPI)
  Zoom levels crop the data coordinates directly — no pixel resampling artifacts
"""

import numpy as np
import yfinance as yf
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
TICKERS = [
    # US Large Cap Tech
    "AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "AMD", "INTC",

    # US Financials
    "JPM", "BAC", "GS", "MS", "WFC",
    "BLK", "C", "AXP", "V", "MA",

    # US Healthcare & Consumer
    "JNJ", "PFE", "UNH", "MRK", "ABBV",
    "PG", "KO", "PEP", "WMT", "COST",

    # US Energy & Industrials
    "XOM", "CVX", "COP", "BA", "CAT",
    "GE", "HON", "LMT", "RTX", "UPS",

    # Commodities & Crypto
    "GC=F", "SI=F", "CL=F", "BTC-USD", "ETH-USD",

    # International ETFs
    "EWJ", "EWZ", "FXI", "EWG", "EWU",
]

START    = "2010-01-01"
END      = "2026-04-11"
Q_VALUES = [1, 10, 50, 75, 100, 150, 200]

OUT_DIR  = r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\plots\phi_full_zoom"
os.makedirs(OUT_DIR, exist_ok=True)

# SVG canvas
W, H   = 8000, 8000
MARGIN = 100

# PNG settings
PNG_DPI     = 300    # 300 DPI × 10 in = 3000×3000 px output
PNG_SIZE_IN = 10     # inches
LINE_WIDTH  = 0.4    # pt — thin enough to reveal fine structure

# Zoom levels: fraction of the full data range shown, centred on the curve's
# robust midpoint. 1.0 = full extent, 0.0625 = 16× zoom into centre.
ZOOM_LEVELS = [
    ("z1_full", 1.00),
    ("z2_2x",   0.50),
    ("z3_4x",   0.25),
    ("z4_8x",   0.125),
    ("z5_16x",  0.0625),
]


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────

def get_returns(ticker):
    df  = yf.download(ticker, start=START, end=END, auto_adjust=True)
    px  = df["Close"].dropna().values.ravel().astype(float)
    px  = px[px > 0]
    ret = np.diff(np.log(px))
    return ret[np.isfinite(ret)]


# ─────────────────────────────────────────────────────────────────────────────
# Φ CURVE
# ─────────────────────────────────────────────────────────────────────────────

def phi(ret, q):
    return np.cumsum(np.exp(1j * q * ret))


# ─────────────────────────────────────────────────────────────────────────────
# BOX-COUNTING FRACTAL DIMENSION
# ─────────────────────────────────────────────────────────────────────────────

def box_counting_dimension(Z, n_scales=20, scale_min=None, scale_max=None):
    xr, yr = Z.real, Z.imag
    span   = max(xr.max() - xr.min(), yr.max() - yr.min(), 1e-12)

    xn = (xr - xr.min()) / span
    yn = (yr - yr.min()) / span

    if scale_max is None: scale_max = 0.20
    if scale_min is None: scale_min = 2.0 / len(Z)

    epsilons = np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales)

    Ns = []
    for eps in epsilons:
        ix       = np.floor(xn / eps).astype(np.int64)
        iy       = np.floor(yn / eps).astype(np.int64)
        occupied = len(set(zip(ix.tolist(), iy.tolist())))
        Ns.append(occupied)

    Ns      = np.array(Ns, dtype=float)
    log_eps = np.log(epsilons)
    log_N   = np.log(Ns)
    valid   = np.isfinite(log_N) & (Ns > 1)

    if valid.sum() < 4:
        return np.nan, epsilons, Ns, np.nan

    coeffs = np.polyfit(log_eps[valid], log_N[valid], 1)
    D_box  = -coeffs[0]

    fitted = np.polyval(coeffs, log_eps[valid])
    ss_res = np.sum((log_N[valid] - fitted) ** 2)
    ss_tot = np.sum((log_N[valid] - log_N[valid].mean()) ** 2)
    r2     = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return D_box, epsilons, Ns, r2


# ─────────────────────────────────────────────────────────────────────────────
# SVG  (original, unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def to_svg_coords(Z, x0, y0, w, h):
    xr, yr = Z.real, Z.imag
    x_lo, x_hi = np.percentile(xr, [1, 99])
    y_lo, y_hi = np.percentile(yr, [1, 99])
    span   = max(x_hi - x_lo, y_hi - y_lo, 1e-12)
    cx, cy = (x_lo + x_hi) / 2, (y_lo + y_hi) / 2
    scale  = min(w, h) / span * 0.98
    sx     = x0 + w / 2 + (xr - cx) * scale
    sy     = y0 + h / 2 - (yr - cy) * scale
    return sx, sy


def make_svg(ticker, Z, q, D_box, r2):
    x0, y0 = MARGIN, MARGIN
    w,  h  = W - 2 * MARGIN, H - 2 * MARGIN
    sx, sy = to_svg_coords(Z, x0, y0, w, h)
    pts    = " L".join(f"{x:.3f},{y:.3f}" for x, y in zip(sx, sy))
    dim_str = (f"D_box = {D_box:.4f}  (R² = {r2:.4f})"
               if np.isfinite(D_box) else "D_box = N/A")

    # Compute where the origin (0,0) maps to in SVG coordinates
    xr, yr = Z.real, Z.imag
    x_lo, x_hi = np.percentile(xr, [1, 99])
    y_lo, y_hi = np.percentile(yr, [1, 99])
    span   = max(x_hi - x_lo, y_hi - y_lo, 1e-12)
    cx, cy = (x_lo + x_hi) / 2, (y_lo + y_hi) / 2
    scale  = min(w, h) / span * 0.98

    # SVG pixel position of the data origin (0, 0)
    ox = x0 + w / 2 + (0 - cx) * scale
    oy = y0 + h / 2 - (0 - cy) * scale

    return f"""<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg"
     width="{W}" height="{H}" viewBox="0 0 {W} {H}">
  <rect width="{W}" height="{H}" fill="white"/>

  <!-- Real axis (horizontal through origin) -->
  <line x1="{x0}" y1="{oy:.2f}" x2="{x0+w}" y2="{oy:.2f}"
        stroke="#aaaaaa" stroke-width="1.5" stroke-dasharray="12,8"/>
  <!-- Imaginary axis (vertical through origin) -->
  <line x1="{ox:.2f}" y1="{y0}" x2="{ox:.2f}" y2="{y0+h}"
        stroke="#aaaaaa" stroke-width="1.5" stroke-dasharray="12,8"/>

  <!-- Axis labels -->
  <text x="{x0+w-10}" y="{oy:.2f}-12" text-anchor="end"
        font-size="22" fill="#888888">Re</text>
  <text x="{ox:.2f}+14" y="{y0+30}"
        font-size="22" fill="#888888">Im</text>

  <path d="M{pts}" stroke="black" stroke-width="0.25" fill="none"/>
  <text x="{W//2}" y="60" text-anchor="middle" font-size="28" fill="black">
    \u03a6(t; q={q}) \u2014 {ticker}
  </text>
  <text x="{W//2}" y="100" text-anchor="middle" font-size="24" fill="#555">
    {dim_str}
  </text>
</svg>"""


# ─────────────────────────────────────────────────────────────────────────────
# MATPLOTLIB PNG — crisp zoomed renders via data-space axis limits
# ─────────────────────────────────────────────────────────────────────────────
def save_zoomed_pngs_mpl(Z, ticker, q, D_box, r2, ticker_dir, base_name):
    xr, yr = Z.real, Z.imag

    x_lo, x_hi = np.percentile(xr, [1, 99])
    y_lo, y_hi = np.percentile(yr, [1, 99])
    cx = (x_lo + x_hi) / 2
    cy = (y_lo + y_hi) / 2
    half_full = max(x_hi - x_lo, y_hi - y_lo) / 2 * 1.02

    dim_str = (f"D_box = {D_box:.4f}   R² = {r2:.4f}"
               if np.isfinite(D_box) else "D_box = N/A")

    for label, frac in ZOOM_LEVELS:
        half = half_full * frac

        fig, ax = plt.subplots(figsize=(PNG_SIZE_IN, PNG_SIZE_IN), dpi=PNG_DPI)
        fig.patch.set_facecolor("white")

        ax.plot(xr, yr,
                color="black",
                linewidth=LINE_WIDTH,
                antialiased=True,
                rasterized=False)

        # ── Re/Im axes + 16x inset on full-extent only ──
        if label == "z1_full":
            ax.axhline(0, color="#aaaaaa", linewidth=1.0,
                       linestyle="--", dashes=(8, 5), zorder=0)
            ax.axvline(0, color="#aaaaaa", linewidth=1.0,
                       linestyle="--", dashes=(8, 5), zorder=0)
            ax.text(cx + half_full * 0.95, 0, r"Re$[\Phi]$",
                    va="bottom", ha="right", fontsize=24, color="#888888")
            ax.text(0, cy + half_full * 0.95, r"Im$[\Phi]$",
                    va="top", ha="left", fontsize=24, color="#888888")

            # Red rectangle on main plot showing where 16x zoom is
            half_16x = half_full * 0.0625
            from matplotlib.patches import Rectangle
            rect = Rectangle((cx - half_16x, cy - half_16x),
                              2 * half_16x, 2 * half_16x,
                              linewidth=1.2, edgecolor="#e63946",
                              facecolor="none", zorder=5)
            ax.add_patch(rect)

            # Inset in bottom-right corner — 16x zoom
            inset_ax = ax.inset_axes([0.67, 0.02, 0.30, 0.30])
            inset_ax.plot(xr, yr,
                          color="black",
                          linewidth=LINE_WIDTH * 0.6,
                          antialiased=True,
                          rasterized=False)
            inset_ax.set_xlim(cx - half_16x, cx + half_16x)
            inset_ax.set_ylim(cy - half_16x, cy + half_16x)
            inset_ax.set_aspect("equal", adjustable="box")
            inset_ax.set_facecolor("#f8f8f8")
            inset_ax.tick_params(left=False, bottom=False,
                                 labelleft=False, labelbottom=False)
            for spine in inset_ax.spines.values():
                spine.set_edgecolor("#e63946")
                spine.set_linewidth(1.2)
            inset_ax.set_title("16× zoom", fontsize=15, color="#e63946",
                   pad=3, fontweight="bold")

        ax.set_xlim(cx - half, cx + half)
        ax.set_ylim(cy - half, cy + half)
        ax.set_aspect("equal", adjustable="box")
        ax.axis("off")

        fig.tight_layout(pad=0.4)
        out_path = os.path.join(ticker_dir, f"{base_name}_{label}.png")
        fig.savefig(out_path, dpi=PNG_DPI, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        print(f"    saved {os.path.relpath(out_path)}")
# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

results = []

header = f"{'Ticker':<12} {'q':>5}  {'D_box':>8}  {'R²':>8}  {'N_pts':>8}"
print(header)
print("-" * len(header))

for ticker in TICKERS:
    print(f"\nProcessing {ticker} …")
    ret = get_returns(ticker)

    if len(ret) < 500:
        print(f"  [SKIP] only {len(ret)} return observations")
        continue

    safe_ticker = ticker.replace("-", "_").replace("=", "_")
    ticker_dir  = os.path.join(OUT_DIR, safe_ticker)
    os.makedirs(ticker_dir, exist_ok=True)

    for q in Q_VALUES:
        Z = phi(ret, q)
        D_box, epsilons, Ns, r2 = box_counting_dimension(Z)

        d_str = f"{D_box:8.4f}" if np.isfinite(D_box) else "     nan"
        r_str = f"{r2:8.4f}"   if np.isfinite(r2)    else "     nan"
        print(f"  {ticker:<10} {q:>5}  {d_str}  {r_str}  {len(Z):>8}")

        results.append({
            'Ticker': ticker,
            'q': q,
            'D_box': D_box,
            'R2': r2,
            'N_pts': len(Z)
        })

        base_name = f"{safe_ticker}_q{q}"

        # SVG
        svg_str  = make_svg(ticker, Z, q, D_box, r2)
        svg_path = os.path.join(ticker_dir, f"{base_name}.svg")
        with open(svg_path, "w", encoding="utf-8") as f:
            f.write(svg_str)

        # PNGs — matplotlib, data-space zoom
        save_zoomed_pngs_mpl(Z, ticker, q, D_box, r2, ticker_dir, base_name)

# Save the results to CSV
df = pd.DataFrame(results)
df.to_csv(r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data\fractal_dimensions.csv", index=False)

print("\nDone. Output in:", OUT_DIR)
print("Structure:")
print("  phi_full_zoom/")
print("    <TICKER>/")
print("      <TICKER>_q<Q>.svg")
print("      <TICKER>_q<Q>_z1_full.png    ← full extent")
print("      <TICKER>_q<Q>_z2_2x.png     ← 2× zoom")
print("      <TICKER>_q<Q>_z3_4x.png     ← 4× zoom")
print("      <TICKER>_q<Q>_z4_8x.png     ← 8× zoom")
print("      <TICKER>_q<Q>_z5_16x.png    ← 16× zoom")
print("CSV file saved: fractal_dimensions.csv")

Ticker           q     D_box        R²     N_pts
------------------------------------------------

Processing AAPL …


[*********************100%***********************]  1 of 1 completed


  AAPL           1    0.9962    1.0000      4091
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q1_z1_full.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q1_z2_2x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q1_z3_4x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q1_z4_8x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q1_z5_16x.png
  AAPL          10    1.0110    0.9999      4091
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q10_z1_full.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q10_z2_2x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q10_z3_4x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q10_z4_8x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q10_z5_16x.png
  AAPL          50    1.0579    0.9998      4091
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q50_z1_full.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q50_z2_2x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q50_z3_4x.png
    saved ..\..\plots\phi_full_zoom\AAPL\AAPL_q50_z4_8x.png
    saved 

[*********************100%***********************]  1 of 1 completed


  MSFT           1    0.9964    1.0000      4091
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q1_z1_full.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q1_z2_2x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q1_z3_4x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q1_z4_8x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q1_z5_16x.png
  MSFT          10    1.0066    0.9999      4091
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q10_z1_full.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q10_z2_2x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q10_z3_4x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q10_z4_8x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q10_z5_16x.png
  MSFT          50    1.0486    0.9997      4091
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q50_z1_full.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q50_z2_2x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q50_z3_4x.png
    saved ..\..\plots\phi_full_zoom\MSFT\MSFT_q50_z4_8x.png
    saved 

...

  FXI            1    0.9958    1.0000      4091
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q1_z1_full.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q1_z2_2x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q1_z3_4x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q1_z4_8x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q1_z5_16x.png
  FXI           10    1.0061    0.9999      4091
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q10_z1_full.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q10_z2_2x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q10_z3_4x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q10_z4_8x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q10_z5_16x.png
  FXI           50    1.0593    0.9993      4091
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q50_z1_full.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q50_z2_2x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q50_z3_4x.png
    saved ..\..\plots\phi_full_zoom\FXI\FXI_q50_z4_8x.png
    saved ..\..\plots\phi_full_zoom\FX

[*********************100%***********************]  1 of 1 completed


  EWG            1    0.9958    1.0000      4091
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q1_z1_full.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q1_z2_2x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q1_z3_4x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q1_z4_8x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q1_z5_16x.png
  EWG           10    1.0056    0.9999      4091
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q10_z1_full.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q10_z2_2x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q10_z3_4x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q10_z4_8x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q10_z5_16x.png
  EWG           50    1.0440    0.9998      4091
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q50_z1_full.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q50_z2_2x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q50_z3_4x.png
    saved ..\..\plots\phi_full_zoom\EWG\EWG_q50_z4_8x.png
    saved ..\..\plots\phi_full_zoom\EW

[*********************100%***********************]  1 of 1 completed


  EWU            1    0.9958    1.0000      4091
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q1_z1_full.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q1_z2_2x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q1_z3_4x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q1_z4_8x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q1_z5_16x.png
  EWU           10    1.0035    0.9999      4091
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q10_z1_full.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q10_z2_2x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q10_z3_4x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q10_z4_8x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q10_z5_16x.png
  EWU           50    1.0375    0.9998      4091
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q50_z1_full.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q50_z2_2x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q50_z3_4x.png
    saved ..\..\plots\phi_full_zoom\EWU\EWU_q50_z4_8x.png
    saved ..\..\plots\phi_full_zoom\EW

In [7]:
df = pd.read_csv(r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data\fractal_dimensions.csv")
print(df['R2'].describe())
print(df[df['R2'] < 0.97])

count    350.000000
mean       0.994556
std        0.014614
min        0.892618
25%        0.997511
50%        0.999620
75%        0.999833
max        0.999959
Name: R2, dtype: float64
      Ticker    q     D_box        R2  N_pts
27      AMZN  200  1.115795  0.954078   4091
33      NVDA  150  1.173376  0.966025   4091
34      NVDA  200  1.077063  0.921565   4091
41      META  200  1.079434  0.942865   3492
47      TSLA  150  1.145350  0.948651   3969
48      TSLA  200  1.018984  0.918247   3969
54      NFLX  150  1.151440  0.967091   4091
55      NFLX  200  1.008945  0.916716   4091
61       AMD  150  0.971991  0.892618   4091
62       AMD  200  0.978721  0.903005   4091
83       BAC  200  1.184534  0.967634   4091
97        MS  200  1.170358  0.963823   4091
230      COP  200  1.144718  0.963488   4091
300     CL=F  200  1.067276  0.927163   4090
313  ETH-USD  150  1.117214  0.952953   3074
314  ETH-USD  200  1.057187  0.919027   3074
328      EWZ  200  1.096582  0.936862   4091


In [ ]:
"""
Block-Bootstrap Confidence Intervals for D_box
================================================
Reuses get_returns(), phi(), box_counting_dimension() from the previous cell.
Computes 95% CIs for D_box on the four deep-dive tickers across all Q_VALUES,
then merges the result into fractal_dimensions.csv.
"""

import numpy as np
import pandas as pd

DEEP_DIVE_TICKERS = TICKERS
N_BOOT     = 500
BLOCK_LEN  = 40
CI_PCTS    = (2.5, 97.5)


def block_bootstrap_dbox(ret, q, n_boot=N_BOOT, block_len=BLOCK_LEN, ci=CI_PCTS):
    n = len(ret)
    n_blocks = int(np.ceil(n / block_len))
    dbox_samples = []

    for _ in range(n_boot):
        starts = np.random.randint(0, n, size=n_blocks)
        blocks = [np.take(ret, range(s, s + block_len), mode='wrap') for s in starts]
        resampled = np.concatenate(blocks)[:n]

        Z = phi(resampled, q)
        D_box, _, _, r2 = box_counting_dimension(Z)
        if np.isfinite(D_box):
            dbox_samples.append(D_box)

    dbox_samples = np.array(dbox_samples)
    if len(dbox_samples) == 0:
        return np.nan, np.nan, np.nan, dbox_samples

    lo, hi = np.percentile(dbox_samples, ci)
    return float(np.mean(dbox_samples)), float(lo), float(hi), dbox_samples


# ── Run bootstrap for deep-dive tickers × all Q_VALUES ──────────────────────
ci_rows = []

for ticker in DEEP_DIVE_TICKERS:
    print(f"\nBootstrapping {ticker} ...")
    ret = get_returns(ticker)

    if len(ret) < 500:
        print(f"  [SKIP] only {len(ret)} return observations")
        continue

    for q in Q_VALUES:
        boot_mean, ci_lo, ci_hi, samples = block_bootstrap_dbox(ret, q)
        width = ci_hi - ci_lo if np.isfinite(ci_hi) else np.nan

        print(f"  q={q:>4}  D_box_boot_mean={boot_mean:.4f}  "
              f"CI=[{ci_lo:.4f}, {ci_hi:.4f}]  width={width:.4f}")

        ci_rows.append({
            "Ticker":          ticker,
            "q":               q,
            "D_box_boot_mean": boot_mean,
            "CI_lo":           ci_lo,
            "CI_hi":           ci_hi,
            "CI_width":        width,
            "n_boot_valid":    len(samples),
        })

# ── Save standalone CI table ─────────────────────────────────────────────
df_ci = pd.DataFrame(ci_rows)
df_ci.to_csv("dbox_bootstrap_ci.csv", index=False)
print("\nSaved → dbox_bootstrap_ci.csv")



[*********************100%***********************]  1 of 1 completed


Bootstrapping AAPL ...


  q=   1  D_box_boot_mean=0.9963  CI=[0.9958, 0.9971]  width=0.0013
  q=  10  D_box_boot_mean=1.0097  CI=[1.0077, 1.0118]  width=0.0041
  q=  50  D_box_boot_mean=1.0563  CI=[1.0495, 1.0623]  width=0.0128
  q=  75  D_box_boot_mean=1.0797  CI=[1.0702, 1.0888]  width=0.0186
  q= 100  D_box_boot_mean=1.1002  CI=[1.0860, 1.1136]  width=0.0276
  q= 150  D_box_boot_mean=1.1390  CI=[1.1182, 1.1595]  width=0.0412


[*********************100%***********************]  1 of 1 completed

  q= 200  D_box_boot_mean=1.1738  CI=[1.1424, 1.2038]  width=0.0614

Bootstrapping MSFT ...


...

  q=   1  D_box_boot_mean=0.9958  CI=[0.9958, 0.9961]  width=0.0003
  q=  10  D_box_boot_mean=1.0055  CI=[1.0030, 1.0079]  width=0.0050
  q=  50  D_box_boot_mean=1.0451  CI=[1.0408, 1.0497]  width=0.0089
  q=  75  D_box_boot_mean=1.0646  CI=[1.0582, 1.0721]  width=0.0138
  q= 100  D_box_boot_mean=1.0817  CI=[1.0727, 1.0910]  width=0.0183
  q= 150  D_box_boot_mean=1.1094  CI=[1.0948, 1.1248]  width=0.0301


[*********************100%***********************]  1 of 1 completed

  q= 200  D_box_boot_mean=1.1363  CI=[1.1191, 1.1549]  width=0.0358

Bootstrapping EWU ...


  q=   1  D_box_boot_mean=0.9958  CI=[0.9958, 0.9961]  width=0.0003
  q=  10  D_box_boot_mean=1.0037  CI=[1.0013, 1.0059]  width=0.0045
  q=  50  D_box_boot_mean=1.0380  CI=[1.0338, 1.0420]  width=0.0083
  q=  75  D_box_boot_mean=1.0547  CI=[1.0492, 1.0605]  width=0.0112
  q= 100  D_box_boot_mean=1.0694  CI=[1.0619, 1.0773]  width=0.0154
  q= 150  D_box_boot_mean=1.0956  CI=[1.0845, 1.1076]  width=0.0230
  q= 200  D_box_boot_mean=1.1155  CI=[1.0988, 1.1345]  width=0.0358

Saved → dbox_bootstrap_ci.csv


FileNotFoundError: [Errno 2] No such file or directory: 'fractal_dimensions.csv'

In [10]:
# ── Merge into fractal_dimensions.csv ────────────────────────────────────
df_main = pd.read_csv(r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data\fractal_dimensions.csv")
df_merged = df_main.merge(df_ci, on=["Ticker", "q"], how="left")
df_merged.to_csv(r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data\fractal_dimensions.csv", index=False)
print("Merged CI columns into fractal_dimensions.csv")

print(df_ci.to_string(index=False))

Merged CI columns into fractal_dimensions.csv
 Ticker   q  D_box_boot_mean    CI_lo    CI_hi  CI_width  n_boot_valid
   AAPL   1         0.996321 0.995815 0.997128  0.001313           500
   AAPL  10         1.009712 1.007708 1.011816  0.004109           500
   AAPL  50         1.056270 1.049530 1.062333  0.012803           500
   AAPL  75         1.079739 1.070185 1.088771  0.018586           500
   AAPL 100         1.100162 1.085993 1.113578  0.027585           500
   AAPL 150         1.139042 1.118238 1.159474  0.041235           500
   AAPL 200         1.173773 1.142411 1.203836  0.061425           500
   MSFT   1         0.996122 0.995791 0.996844  0.001054           500
   MSFT  10         1.007690 1.005200 1.009917  0.004717           500
   MSFT  50         1.051023 1.045300 1.056546  0.011247           500
   MSFT  75         1.072683 1.065016 1.080550  0.015534           500
   MSFT 100         1.093807 1.083421 1.104855  0.021435           500
   MSFT 150         1.135593 1.